# 200 — TCGA Multi-omic Candidate Cohort Construction

## Objective

Construct a reproducible candidate cohort of TCGA primary tumors with sample-matched RNA-seq and DNA methylation data.

This notebook operates at the metadata and file-index levels. It identifies eligible multi-omic pairs before RNA-seq or methylation quality control, feature harmonization, matrix construction, or biological modeling.

## Input cohorts

The analysis uses the frozen and download-validated TCGA cohorts generated during Phase 1:

- primary-tumor RNA-seq quantified with the GDC STAR Counts workflow;
- primary-tumor DNA methylation beta values processed with SeSAMe;
- HM27 and HM450 methylation platforms only.

The version-controlled file indices define the authoritative input cohorts. RNA-seq GDC metadata is additionally used to recover aliquot-level provenance that is not present in the frozen RNA-seq file index.

## Biological unit and matching policy

The independent biological unit is the TCGA case.

The following rules apply:

- each final candidate observation must represent one `case_id`;
- RNA-seq and methylation must originate from the same `sample_id`;
- files from different samples belonging to the same case are not treated as a multi-omic pair;
- aliquot and file identifiers are retained for technical provenance;
- a case must not contribute more than one independent observation;
- no duplicate or ambiguous candidate is resolved arbitrarily.

## File-selection policy

For each sample:

- a single eligible file per modality is selected directly;
- when both HM27 and HM450 methylation files are available, HM450 is prioritized;
- multiple files from the same modality and platform remain unresolved candidates;
- cases with multiple eligible sample-level pairs remain ambiguous at the case level;
- unambiguous pairs and ambiguous candidates are retained separately.

These rules define metadata-level eligibility only. Molecular quality control may subsequently exclude or replace candidate files.

## RNA-seq representation policy

The `unstranded` STAR Counts column is the canonical expression input.

This notebook records that decision but does not construct or normalize the expression matrix. Gene filtering, library normalization, transformation, and the canonical gene-identifier policy will be defined empirically during RNA-seq quality control.

TPM, FPKM, and FPKM-UQ are not used as the primary expression representation.

## Scope

This notebook will:

- load and validate the frozen RNA-seq and methylation file indices;
- recover RNA-seq aliquot identifiers from GDC metadata;
- harmonize modality-specific identifier names;
- quantify case-, sample-, and file-level overlap;
- characterize multiplicity by modality and methylation platform;
- apply the confirmed metadata-level selection policy;
- construct an unambiguous sample-matched candidate cohort;
- retain unresolved candidates in a separate table;
- summarize coverage by TCGA project and methylation platform.

This notebook will not:

- load complete molecular matrices;
- perform RNA-seq or methylation quality control;
- filter genes or methylation probes;
- normalize expression or methylation values;
- harmonize HM27 and HM450 probe spaces;
- resolve same-platform technical duplicates using arbitrary rules;
- perform methylation–expression integration;
- discover biological programs;
- infer causality or clinical resistance.

## Outputs

After the empirical overlap and ambiguity structure has been verified, this notebook will write only downstream-consumable artifacts:

- a case-level table of unambiguous, sample-matched candidate pairs;
- a separate table retaining ambiguous candidates for downstream quality-control resolution.

Coverage summaries will remain within the notebook unless a downstream analysis requires them. Exact output schemas and filenames will be fixed after validating the observed candidate structure.

In [1]:
# =============================================================================
# 200 — TCGA Multi-omic Candidate Cohort Construction
# Imports and input paths
# =============================================================================

import json
import pandas as pd


# -----------------------------------------------------------------------------
# Project utilities
# -----------------------------------------------------------------------------

from pancancer_epigenetics.utils.paths import Paths, project_relative_path


# -----------------------------------------------------------------------------
# Canonical project root
# -----------------------------------------------------------------------------

PROJECT_ROOT = Paths.root


# -----------------------------------------------------------------------------
# Frozen TCGA RNA-seq inputs
# -----------------------------------------------------------------------------

RNA_MANIFEST_DIR = (
    Paths.config
    / "manifests"
    / "tcga_rna"
)

RNA_FILE_INDEX_PATH = (
    RNA_MANIFEST_DIR
    / "gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv"
)

RNA_METADATA_PATH = (
    RNA_MANIFEST_DIR
    / "gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json"
)

RNA_COHORT_FREEZE_PATH = (
    RNA_MANIFEST_DIR
    / "gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json"
)


# -----------------------------------------------------------------------------
# Frozen TCGA methylation inputs
# -----------------------------------------------------------------------------

METHYLATION_MANIFEST_DIR = (
    Paths.config
    / "manifests"
    / "tcga_methylation"
)

METHYLATION_FILE_INDEX_PATH = (
    METHYLATION_MANIFEST_DIR
    / (
        "gdc_file_index_tcga_primary_tumor_methylation_array_"
        "sesame_beta_values_hm27_hm450.tsv"
    )
)

METHYLATION_COHORT_FREEZE_PATH = (
    METHYLATION_MANIFEST_DIR
    / (
        "gdc_cohort_freeze_tcga_primary_tumor_methylation_array_"
        "sesame_beta_values_hm27_hm450.json"
    )
)


# -----------------------------------------------------------------------------
# Path summary
# -----------------------------------------------------------------------------

print("TCGA multi-omic cohort-construction inputs resolved.")
print(
    f"Project root:               "
    f"{project_relative_path(PROJECT_ROOT)}"
)
print(
    f"RNA-seq file index:         "
    f"{project_relative_path(RNA_FILE_INDEX_PATH)}"
)
print(
    f"RNA-seq GDC metadata:       "
    f"{project_relative_path(RNA_METADATA_PATH)}"
)
print(
    f"RNA-seq cohort freeze:      "
    f"{project_relative_path(RNA_COHORT_FREEZE_PATH)}"
)
print(
    f"Methylation file index:     "
    f"{project_relative_path(METHYLATION_FILE_INDEX_PATH)}"
)
print(
    f"Methylation cohort freeze:  "
    f"{project_relative_path(METHYLATION_COHORT_FREEZE_PATH)}"
)

TCGA multi-omic cohort-construction inputs resolved.
Project root:               .
RNA-seq file index:         config/manifests/tcga_rna/gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv
RNA-seq GDC metadata:       config/manifests/tcga_rna/gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json
RNA-seq cohort freeze:      config/manifests/tcga_rna/gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json
Methylation file index:     config/manifests/tcga_methylation/gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv
Methylation cohort freeze:  config/manifests/tcga_methylation/gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json


In [2]:
# =============================================================================
# Validate input availability and load frozen cohort artifacts
# =============================================================================

input_paths = {
    "RNA-seq file index": RNA_FILE_INDEX_PATH,
    "RNA-seq GDC metadata": RNA_METADATA_PATH,
    "RNA-seq cohort freeze": RNA_COHORT_FREEZE_PATH,
    "Methylation file index": METHYLATION_FILE_INDEX_PATH,
    "Methylation cohort freeze": METHYLATION_COHORT_FREEZE_PATH,
}


# -----------------------------------------------------------------------------
# Validate input paths
# -----------------------------------------------------------------------------

missing_inputs = [
    path
    for path in input_paths.values()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required TCGA cohort inputs:\n"
        + "\n".join(
            project_relative_path(path)
            for path in missing_inputs
        )
    )

non_file_inputs = [
    path
    for path in input_paths.values()
    if not path.is_file()
]

if non_file_inputs:
    raise ValueError(
        "The following TCGA inputs are not files:\n"
        + "\n".join(
            project_relative_path(path)
            for path in non_file_inputs
        )
    )


# -----------------------------------------------------------------------------
# Load frozen file indices
# -----------------------------------------------------------------------------

rna_file_index = pd.read_csv(
    RNA_FILE_INDEX_PATH,
    sep="\t",
    dtype="string",
)

methylation_file_index = pd.read_csv(
    METHYLATION_FILE_INDEX_PATH,
    sep="\t",
    dtype="string",
)

rna_file_index["file_size_bytes"] = pd.to_numeric(
    rna_file_index["file_size_bytes"],
    errors="raise",
).astype("int64")

methylation_file_index["file_size_bytes"] = pd.to_numeric(
    methylation_file_index["file_size_bytes"],
    errors="raise",
).astype("int64")


# -----------------------------------------------------------------------------
# Load GDC metadata and cohort-freeze records
# -----------------------------------------------------------------------------

with RNA_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    rna_gdc_metadata = json.load(handle)

with RNA_COHORT_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    rna_cohort_freeze = json.load(handle)

with METHYLATION_COHORT_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    methylation_cohort_freeze = json.load(handle)


# -----------------------------------------------------------------------------
# Load summary
# -----------------------------------------------------------------------------

print("Frozen TCGA cohort artifacts loaded.")
print(
    f"RNA-seq file index:        "
    f"{rna_file_index.shape}"
)
print(
    f"RNA-seq metadata records:  "
    f"{len(rna_gdc_metadata):,}"
)
print(
    f"Methylation file index:    "
    f"{methylation_file_index.shape}"
)

Frozen TCGA cohort artifacts loaded.
RNA-seq file index:        (10308, 14)
RNA-seq metadata records:  10,308
Methylation file index:    (10897, 25)


In [3]:
# =============================================================================
# Validate frozen cohort structures and provenance consistency
# =============================================================================

required_rna_columns = {
    "file_id",
    "file_name",
    "md5",
    "file_size_bytes",
    "state",
    "project_id",
    "case_id",
    "sample_id",
    "data_category",
    "data_type",
    "tissue_type",
    "tumor_descriptor",
    "specimen_type",
    "preservation_method",
}

required_methylation_columns = {
    "file_id",
    "file_name",
    "md5",
    "file_size_bytes",
    "state",
    "project_id",
    "case_uuid",
    "case_submitter_id",
    "sample_submitter_id",
    "aliquot_uuid",
    "aliquot_submitter_id",
    "data_category",
    "data_type",
    "experimental_strategy",
    "platform",
    "workflow_type",
    "tissue_type",
    "tumor_descriptor",
}

missing_rna_columns = (
    required_rna_columns
    - set(rna_file_index.columns)
)

missing_methylation_columns = (
    required_methylation_columns
    - set(methylation_file_index.columns)
)

if missing_rna_columns:
    raise ValueError(
        "Missing RNA-seq file-index columns: "
        f"{sorted(missing_rna_columns)}"
    )

if missing_methylation_columns:
    raise ValueError(
        "Missing methylation file-index columns: "
        f"{sorted(missing_methylation_columns)}"
    )


# -----------------------------------------------------------------------------
# Validate key identifiers
# -----------------------------------------------------------------------------

key_columns = {
    "RNA-seq": (
        rna_file_index,
        [
            "file_id",
            "project_id",
            "case_id",
            "sample_id",
        ],
    ),
    "Methylation": (
        methylation_file_index,
        [
            "file_id",
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "aliquot_submitter_id",
        ],
    ),
}

for modality, (dataframe, columns) in key_columns.items():
    missing_key_values = (
        dataframe[columns]
        .isna()
        .sum()
    )

    missing_key_values = missing_key_values[
        missing_key_values > 0
    ]

    if not missing_key_values.empty:
        raise ValueError(
            f"{modality} contains missing key identifiers: "
            f"{missing_key_values.to_dict()}"
        )


# -----------------------------------------------------------------------------
# Validate file-level uniqueness and frozen counts
# -----------------------------------------------------------------------------

expected_rna_files = int(
    rna_cohort_freeze[
        "cohort_counts"
    ]["n_files"]
)

expected_methylation_files = int(
    methylation_cohort_freeze[
        "frozen_cohort_counts"
    ]["n_files"]
)

rna_metadata_file_ids = [
    record["file_id"]
    for record in rna_gdc_metadata
]

EXPECTED_METHYLATION_PLATFORMS = {
    "Illumina Human Methylation 27",
    "Illumina Human Methylation 450",
}

checks = {
    "rna_expected_file_count": (
        len(rna_file_index)
        == expected_rna_files
    ),
    "methylation_expected_file_count": (
        len(methylation_file_index)
        == expected_methylation_files
    ),
    "rna_unique_file_ids": (
        not rna_file_index["file_id"].duplicated().any()
    ),
    "methylation_unique_file_ids": (
        not methylation_file_index["file_id"].duplicated().any()
    ),
    "rna_unique_metadata_file_ids": (
        len(rna_metadata_file_ids)
        == len(set(rna_metadata_file_ids))
    ),
    "rna_index_metadata_file_id_match": (
        set(rna_file_index["file_id"])
        == set(rna_metadata_file_ids)
    ),
    "rna_all_released": (
        rna_file_index["state"] == "released"
    ).all(),
    "methylation_all_released": (
        methylation_file_index["state"] == "released"
    ).all(),
    "rna_all_primary_tumor": (
        (rna_file_index["tissue_type"] == "Tumor")
        & (
            rna_file_index["tumor_descriptor"]
            == "Primary"
        )
    ).all(),
    "methylation_all_primary_tumor": (
        (
            methylation_file_index["tissue_type"]
            == "Tumor"
        )
        & (
            methylation_file_index["tumor_descriptor"]
            == "Primary"
        )
    ).all(),
    "rna_all_star_counts": all(
        record.get("analysis", {}).get("workflow_type")
        == "STAR - Counts"
        for record in rna_gdc_metadata
    ),
    "methylation_expected_platforms": (
        set(methylation_file_index["platform"].unique())
        == EXPECTED_METHYLATION_PLATFORMS
    ),
}

for check_name, passed in checks.items():
    print(f"{check_name}: {passed}")

failed_checks = [
    check_name
    for check_name, passed in checks.items()
    if not passed
]

if failed_checks:
    raise ValueError(
        "Frozen input validation failed: "
        + ", ".join(failed_checks)
    )


# -----------------------------------------------------------------------------
# Input cohort summary
# -----------------------------------------------------------------------------

input_cohort_summary = pd.DataFrame(
    [
        {
            "modality": "RNA-seq",
            "n_files": rna_file_index["file_id"].nunique(),
            "n_samples": rna_file_index["sample_id"].nunique(),
            "n_cases": rna_file_index["case_id"].nunique(),
            "n_projects": rna_file_index["project_id"].nunique(),
            "platform_scope": "STAR - Counts",
        },
        {
            "modality": "DNA methylation",
            "n_files": methylation_file_index[
                "file_id"
            ].nunique(),
            "n_samples": methylation_file_index[
                "sample_submitter_id"
            ].nunique(),
            "n_cases": methylation_file_index[
                "case_submitter_id"
            ].nunique(),
            "n_projects": methylation_file_index[
                "project_id"
            ].nunique(),
            "platform_scope": "HM27 + HM450",
        },
    ]
)

print()
print("Frozen input cohorts validated.")
display(input_cohort_summary)

rna_expected_file_count: True
methylation_expected_file_count: True
rna_unique_file_ids: True
methylation_unique_file_ids: True
rna_unique_metadata_file_ids: True
rna_index_metadata_file_id_match: True
rna_all_released: True
methylation_all_released: True
rna_all_primary_tumor: True
methylation_all_primary_tumor: True
rna_all_star_counts: True
methylation_expected_platforms: True

Frozen input cohorts validated.


,modality,n_files,n_samples,n_cases,n_projects,platform_scope
0,RNA-seq,10308,10174,10122,33,STAR - Counts
1,DNA methylation,10897,10656,10610,33,HM27 + HM450


## RNA-seq aliquot-level provenance

The frozen RNA-seq file index contains project-, case-, and sample-level identifiers but does not retain aliquot identifiers. Aliquot-level provenance is therefore recovered from the `associated_entities` field of the authoritative GDC metadata.

For this metadata export, each file is linked directly to its associated biological entity:

`file → associated entity`

For aliquot-associated records:

- `associated_entities[].case_id` is the case UUID;
- `associated_entities[].entity_id` is the aliquot UUID;
- `associated_entities[].entity_submitter_id` is the full TCGA aliquot barcode;
- `associated_entities[].entity_type` identifies the entity as an aliquot.

The case and sample submitter identifiers are reconstructed deterministically from the structured TCGA aliquot barcode and validated against the frozen RNA-seq file index. Project identifiers are retained from the frozen index because they are not included in the metadata association.

The optional `annotations` field is not used for provenance reconstruction because it contains administrative annotations for only a subset of records and does not represent the file-to-biospecimen hierarchy.

This step:

- does not select or discard molecular files;
- preserves file, case, sample, and aliquot provenance;
- quantifies file-to-aliquot multiplicity;
- validates the identifier representation used by the frozen index;
- retains any observed multiplicity for later ambiguity handling.

In [4]:
# =============================================================================
# Recover RNA-seq aliquot-level provenance from flattened GDC metadata
# =============================================================================

associated_entity_columns = [
    "file_id",
    "case_uuid",
    "aliquot_uuid",
    "aliquot_submitter_id",
    "entity_type",
]

associated_entity_rows = []

for file_record in rna_gdc_metadata:
    file_id = file_record.get("file_id")

    for entity_record in (
        file_record.get("associated_entities") or []
    ):
        associated_entity_rows.append(
            {
                "file_id": file_id,
                "case_uuid": entity_record.get("case_id"),
                "aliquot_uuid": entity_record.get("entity_id"),
                "aliquot_submitter_id": (
                    entity_record.get("entity_submitter_id")
                ),
                "entity_type": entity_record.get("entity_type"),
            }
        )

rna_associated_entities = pd.DataFrame.from_records(
    associated_entity_rows,
    columns=associated_entity_columns,
).astype("string")


# -----------------------------------------------------------------------------
# Diagnose file-to-associated-entity multiplicity
# -----------------------------------------------------------------------------

rna_index_file_ids = set(
    rna_file_index["file_id"].dropna()
)

rna_associated_file_ids = set(
    rna_associated_entities["file_id"].dropna()
)

associated_entities_per_file = (
    rna_associated_entities
    .groupby("file_id")
    .size()
    .reindex(rna_file_index["file_id"])
    .fillna(0)
    .astype("int64")
)

rna_entity_multiplicity = (
    associated_entities_per_file
    .value_counts()
    .sort_index()
    .rename_axis("n_associated_entities")
    .rename("n_files")
    .reset_index()
)

files_without_associated_entities = sorted(
    rna_index_file_ids
    - rna_associated_file_ids
)

unexpected_associated_file_ids = sorted(
    rna_associated_file_ids
    - rna_index_file_ids
)

files_with_multiple_associated_entities = (
    associated_entities_per_file[
        associated_entities_per_file > 1
    ]
)


# -----------------------------------------------------------------------------
# Parse TCGA case and sample submitter identifiers from aliquot barcodes
# -----------------------------------------------------------------------------

def tcga_barcode_prefix(
    barcode,
    n_components,
):
    if pd.isna(barcode):
        return pd.NA

    components = str(barcode).split("-")

    if (
        len(components) < n_components
        or components[0] != "TCGA"
        or any(
            component == ""
            for component in components[:n_components]
        )
    ):
        return pd.NA

    return "-".join(
        components[:n_components]
    )


rna_associated_entities[
    "case_submitter_id_from_aliquot"
] = (
    rna_associated_entities["aliquot_submitter_id"]
    .map(
        lambda barcode: tcga_barcode_prefix(
            barcode,
            n_components=3,
        )
    )
    .astype("string")
)

rna_associated_entities[
    "sample_submitter_id_from_aliquot"
] = (
    rna_associated_entities["aliquot_submitter_id"]
    .map(
        lambda barcode: tcga_barcode_prefix(
            barcode,
            n_components=4,
        )
    )
    .astype("string")
)


# -----------------------------------------------------------------------------
# Join frozen-index identifiers to recovered aliquot provenance
# -----------------------------------------------------------------------------

rna_provenance_diagnostic = (
    rna_file_index[
        [
            "file_id",
            "project_id",
            "case_id",
            "sample_id",
        ]
    ]
    .rename(
        columns={
            "case_id": "index_case_id",
            "sample_id": "index_sample_id",
        }
    )
    .merge(
        rna_associated_entities,
        on="file_id",
        how="left",
        validate="one_to_many",
    )
)


# -----------------------------------------------------------------------------
# Determine the identifier representation used by the frozen index
# -----------------------------------------------------------------------------

case_matches_metadata_uuid = (
    rna_provenance_diagnostic["index_case_id"]
    .eq(
        rna_provenance_diagnostic["case_uuid"]
    )
    .fillna(False)
    .all()
)

case_matches_submitter_id = (
    rna_provenance_diagnostic["index_case_id"]
    .eq(
        rna_provenance_diagnostic[
            "case_submitter_id_from_aliquot"
        ]
    )
    .fillna(False)
    .all()
)

sample_matches_submitter_id = (
    rna_provenance_diagnostic["index_sample_id"]
    .eq(
        rna_provenance_diagnostic[
            "sample_submitter_id_from_aliquot"
        ]
    )
    .fillna(False)
    .all()
)

rna_identifier_alignment = pd.DataFrame(
    [
        {
            "index_column": "case_id",
            "candidate_representation": "GDC case UUID",
            "matches_all_rows": case_matches_metadata_uuid,
        },
        {
            "index_column": "case_id",
            "candidate_representation": (
                "TCGA case submitter ID from aliquot barcode"
            ),
            "matches_all_rows": case_matches_submitter_id,
        },
        {
            "index_column": "sample_id",
            "candidate_representation": (
                "TCGA sample submitter ID from aliquot barcode"
            ),
            "matches_all_rows": sample_matches_submitter_id,
        },
    ]
)


# -----------------------------------------------------------------------------
# Validate recovered provenance
# -----------------------------------------------------------------------------

required_association_columns = [
    "file_id",
    "case_uuid",
    "aliquot_uuid",
    "aliquot_submitter_id",
    "entity_type",
    "case_submitter_id_from_aliquot",
    "sample_submitter_id_from_aliquot",
]

missing_association_values = (
    rna_associated_entities[
        required_association_columns
    ]
    .isna()
    .sum()
)

missing_association_values = missing_association_values[
    missing_association_values > 0
]

case_identifier_alignment_resolved = (
    case_matches_metadata_uuid
    or case_matches_submitter_id
)

sample_identifier_alignment_resolved = (
    sample_matches_submitter_id
)

rna_provenance_checks = {
    "provenance_table_is_not_empty": (
        not rna_associated_entities.empty
    ),
    "all_index_files_have_associated_entities": (
        not files_without_associated_entities
    ),
    "provenance_has_no_unexpected_files": (
        not unexpected_associated_file_ids
    ),
    "required_association_identifiers_complete": (
        missing_association_values.empty
    ),
    "all_associated_entities_are_aliquots": (
        rna_associated_entities["entity_type"]
        .eq("aliquot")
        .all()
    ),
    "case_identifier_alignment_resolved": (
        case_identifier_alignment_resolved
    ),
    "sample_identifier_alignment_resolved": (
        sample_identifier_alignment_resolved
    ),
}


# -----------------------------------------------------------------------------
# Construct the standardized RNA-seq provenance table
# -----------------------------------------------------------------------------

rna_aliquot_provenance = (
    rna_provenance_diagnostic[
        [
            "file_id",
            "project_id",
            "case_uuid",
            "case_submitter_id_from_aliquot",
            "sample_submitter_id_from_aliquot",
            "aliquot_uuid",
            "aliquot_submitter_id",
        ]
    ]
    .rename(
        columns={
            "case_submitter_id_from_aliquot": (
                "case_submitter_id"
            ),
            "sample_submitter_id_from_aliquot": (
                "sample_submitter_id"
            ),
        }
    )
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "aliquot_submitter_id",
            "file_id",
        ],
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

print("RNA-seq aliquot provenance recovered.")
print(
    f"RNA-seq files in index:             "
    f"{len(rna_file_index):,}"
)
print(
    f"Associated-entity rows:             "
    f"{len(rna_associated_entities):,}"
)
print(
    f"Files represented in provenance:    "
    f"{rna_associated_entities['file_id'].nunique():,}"
)
print(
    f"Files without associated entities:  "
    f"{len(files_without_associated_entities):,}"
)
print(
    f"Files with multiple entities:       "
    f"{len(files_with_multiple_associated_entities):,}"
)

print()
print("File-to-associated-entity multiplicity:")
display(rna_entity_multiplicity)

print("Frozen-index identifier alignment:")
display(rna_identifier_alignment)

print("Provenance integrity checks:")
for check_name, passed in rna_provenance_checks.items():
    print(f"{check_name}: {passed}")

failed_provenance_checks = [
    check_name
    for check_name, passed in rna_provenance_checks.items()
    if not passed
]

if failed_provenance_checks:
    raise ValueError(
        "RNA-seq provenance validation failed: "
        + ", ".join(failed_provenance_checks)
    )

print()
print("Standardized RNA-seq provenance:")
print(f"Shape: {rna_aliquot_provenance.shape}")
display(rna_aliquot_provenance.head())

RNA-seq aliquot provenance recovered.
RNA-seq files in index:             10,308
Associated-entity rows:             10,308
Files represented in provenance:    10,308
Files without associated entities:  0
Files with multiple entities:       0

File-to-associated-entity multiplicity:


,n_associated_entities,n_files
0,1,10308


Frozen-index identifier alignment:


,index_column,candidate_representation,matches_all_rows
0,case_id,GDC case UUID,False
1,case_id,TCGA case submitter ID from aliquot barcode,True
2,sample_id,TCGA sample submitter ID from aliquot barcode,True


Provenance integrity checks:
provenance_table_is_not_empty: True
all_index_files_have_associated_entities: True
provenance_has_no_unexpected_files: True
required_association_identifiers_complete: True
all_associated_entities_are_aliquots: True
case_identifier_alignment_resolved: True
sample_identifier_alignment_resolved: True

Standardized RNA-seq provenance:
Shape: (10308, 7)


,file_id,project_id,case_uuid,case_submitter_id,sample_submitter_id,aliquot_uuid,aliquot_submitter_id
0,fe16b2d3-17b0-4e24-ab31-62d2e951b3a2,TCGA-ACC,b3164f7b-c826-4e08-9ee6-8ff96d29b913,TCGA-OR-A5J1,TCGA-OR-A5J1-01A,37e88158-0743-45b8-87cf-1d7fe878527f,TCGA-OR-A5J1-01A-11R-A29S-07
1,f11aa62e-5d67-4e49-8f64-5ed663a3b424,TCGA-ACC,8e7c2e31-d085-4b75-a970-162526dd07a0,TCGA-OR-A5J2,TCGA-OR-A5J2-01A,421bbd6b-fb2a-4a6d-a298-057c1b65f041,TCGA-OR-A5J2-01A-11R-A29S-07
2,3137ba2b-0d6b-4fcc-8512-894fd6a43ff6,TCGA-ACC,dfd687bc-6e69-42f7-af94-d17fc150d1a1,TCGA-OR-A5J3,TCGA-OR-A5J3-01A,83b02107-0c7c-450d-84aa-9ed4e1d755c5,TCGA-OR-A5J3-01A-11R-A29S-07
3,b8abd8a3-1525-4bf5-b083-eddefa3ca19e,TCGA-ACC,802dbd0d-ef07-4c91-ab8d-1dd39532e947,TCGA-OR-A5J5,TCGA-OR-A5J5-01A,f72bfbe6-411d-412e-aaab-1a2414e544ec,TCGA-OR-A5J5-01A-11R-A29S-07
4,224fa76c-cf8d-48d8-8723-15907a04f2d2,TCGA-ACC,c8898b42-b704-45a0-9829-144b98f416e0,TCGA-OR-A5J6,TCGA-OR-A5J6-01A,bb6e0e73-ff48-4627-bce6-f672c06b7de4,TCGA-OR-A5J6-01A-31R-A29S-07


## Cross-modality identifier harmonization

RNA-seq and DNA methylation provenance are standardized into modality-specific, pairing-ready file indexes using the same TCGA identifier hierarchy.

The identifiers serve different purposes:

- `case_submitter_id` identifies the participant and defines the case-level analytical unit;
- `sample_submitter_id` identifies the exact biological sample and will be required for direct cross-modality pairing;
- aliquot identifiers preserve assay-specific biospecimen provenance but are not expected to match across molecular modalities;
- GDC case UUIDs provide an independent cross-modality identity check;
- `project_id` preserves tumor-project membership.

A shared case alone is not sufficient to establish a molecular pair. Direct RNA-seq–methylation candidate pairs will subsequently require an exact match on:

`project_id + case_submitter_id + sample_submitter_id`

This prevents different primary-tumor samples or vials from the same participant from being silently treated as the same biospecimen.

At this stage:

- no files are selected or discarded;
- no preference is assigned between HM27 and HM450;
- no candidate pairs are constructed;
- no output artifacts are written.

In [5]:
# =============================================================================
# Harmonize RNA-seq and methylation provenance schemas
# =============================================================================

rna_pairing_index = (
    rna_aliquot_provenance
    .merge(
        rna_file_index[
            [
                "file_id",
                "file_name",
            ]
        ],
        on="file_id",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "case_uuid": "rna_case_uuid",
            "aliquot_uuid": "rna_aliquot_uuid",
            "aliquot_submitter_id": (
                "rna_aliquot_submitter_id"
            ),
            "file_id": "rna_file_id",
            "file_name": "rna_file_name",
        }
    )
    [
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "rna_case_uuid",
            "rna_aliquot_uuid",
            "rna_aliquot_submitter_id",
            "rna_file_id",
            "rna_file_name",
        ]
    ]
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "rna_aliquot_submitter_id",
            "rna_file_id",
        ],
        kind="stable",
        ignore_index=True,
    )
)


methylation_pairing_index = (
    methylation_file_index[
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "case_uuid",
            "aliquot_uuid",
            "aliquot_submitter_id",
            "file_id",
            "file_name",
            "platform",
        ]
    ]
    .rename(
        columns={
            "case_uuid": "methylation_case_uuid",
            "aliquot_uuid": "methylation_aliquot_uuid",
            "aliquot_submitter_id": (
                "methylation_aliquot_submitter_id"
            ),
            "file_id": "methylation_file_id",
            "file_name": "methylation_file_name",
            "platform": "methylation_platform",
        }
    )
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "methylation_platform",
            "methylation_aliquot_submitter_id",
            "methylation_file_id",
        ],
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Validate standardized identifiers
# -----------------------------------------------------------------------------

rna_required_pairing_columns = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "rna_case_uuid",
    "rna_aliquot_uuid",
    "rna_aliquot_submitter_id",
    "rna_file_id",
    "rna_file_name",
]

methylation_required_pairing_columns = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "methylation_case_uuid",
    "methylation_aliquot_uuid",
    "methylation_aliquot_submitter_id",
    "methylation_file_id",
    "methylation_file_name",
    "methylation_platform",
]


rna_case_from_aliquot = (
    rna_pairing_index["rna_aliquot_submitter_id"]
    .map(
        lambda barcode: tcga_barcode_prefix(
            barcode,
            n_components=3,
        )
    )
    .astype("string")
)

rna_sample_from_aliquot = (
    rna_pairing_index["rna_aliquot_submitter_id"]
    .map(
        lambda barcode: tcga_barcode_prefix(
            barcode,
            n_components=4,
        )
    )
    .astype("string")
)

methylation_case_from_aliquot = (
    methylation_pairing_index[
        "methylation_aliquot_submitter_id"
    ]
    .map(
        lambda barcode: tcga_barcode_prefix(
            barcode,
            n_components=3,
        )
    )
    .astype("string")
)

methylation_sample_from_aliquot = (
    methylation_pairing_index[
        "methylation_aliquot_submitter_id"
    ]
    .map(
        lambda barcode: tcga_barcode_prefix(
            barcode,
            n_components=4,
        )
    )
    .astype("string")
)


pairing_index_checks = {
    "rna_expected_row_count": (
        len(rna_pairing_index)
        == len(rna_file_index)
    ),
    "methylation_expected_row_count": (
        len(methylation_pairing_index)
        == len(methylation_file_index)
    ),
    "rna_unique_file_ids": (
        not rna_pairing_index["rna_file_id"]
        .duplicated()
        .any()
    ),
    "methylation_unique_file_ids": (
        not methylation_pairing_index[
            "methylation_file_id"
        ]
        .duplicated()
        .any()
    ),
    "rna_required_identifiers_complete": (
        rna_pairing_index[
            rna_required_pairing_columns
        ]
        .notna()
        .all()
        .all()
    ),
    "methylation_required_identifiers_complete": (
        methylation_pairing_index[
            methylation_required_pairing_columns
        ]
        .notna()
        .all()
        .all()
    ),
    "rna_case_barcode_matches_aliquot": (
        rna_pairing_index["case_submitter_id"]
        .eq(rna_case_from_aliquot)
        .fillna(False)
        .all()
    ),
    "rna_sample_barcode_matches_aliquot": (
        rna_pairing_index["sample_submitter_id"]
        .eq(rna_sample_from_aliquot)
        .fillna(False)
        .all()
    ),
    "methylation_case_barcode_matches_aliquot": (
        methylation_pairing_index[
            "case_submitter_id"
        ]
        .eq(methylation_case_from_aliquot)
        .fillna(False)
        .all()
    ),
    "methylation_sample_barcode_matches_aliquot": (
        methylation_pairing_index[
            "sample_submitter_id"
        ]
        .eq(methylation_sample_from_aliquot)
        .fillna(False)
        .all()
    ),
    "rna_one_project_per_case_barcode": (
        rna_pairing_index
        .groupby("case_submitter_id")["project_id"]
        .nunique()
        .le(1)
        .all()
    ),
    "methylation_one_project_per_case_barcode": (
        methylation_pairing_index
        .groupby("case_submitter_id")["project_id"]
        .nunique()
        .le(1)
        .all()
    ),
    "rna_one_case_uuid_per_case_barcode": (
        rna_pairing_index
        .groupby("case_submitter_id")["rna_case_uuid"]
        .nunique()
        .le(1)
        .all()
    ),
    "methylation_one_case_uuid_per_case_barcode": (
        methylation_pairing_index
        .groupby("case_submitter_id")[
            "methylation_case_uuid"
        ]
        .nunique()
        .le(1)
        .all()
    ),
}


print("Pairing-index integrity checks:")

for check_name, passed in pairing_index_checks.items():
    print(f"{check_name}: {passed}")

failed_pairing_index_checks = [
    check_name
    for check_name, passed in pairing_index_checks.items()
    if not passed
]

if failed_pairing_index_checks:
    raise ValueError(
        "Pairing-index harmonization failed: "
        + ", ".join(failed_pairing_index_checks)
    )


# -----------------------------------------------------------------------------
# Compare case identities across modalities
# -----------------------------------------------------------------------------

rna_case_identity = (
    rna_pairing_index[
        [
            "case_submitter_id",
            "project_id",
            "rna_case_uuid",
        ]
    ]
    .drop_duplicates()
)

methylation_case_identity = (
    methylation_pairing_index[
        [
            "case_submitter_id",
            "project_id",
            "methylation_case_uuid",
        ]
    ]
    .drop_duplicates()
)

shared_case_identity = (
    rna_case_identity
    .merge(
        methylation_case_identity,
        on="case_submitter_id",
        how="inner",
        suffixes=(
            "_rna",
            "_methylation",
        ),
        validate="one_to_one",
    )
)

shared_case_project_matches = (
    shared_case_identity["project_id_rna"]
    .eq(
        shared_case_identity[
            "project_id_methylation"
        ]
    )
)

shared_case_uuid_matches = (
    shared_case_identity["rna_case_uuid"]
    .eq(
        shared_case_identity[
            "methylation_case_uuid"
        ]
    )
)

cross_modality_case_checks = {
    "shared_case_identity_is_not_empty": (
        not shared_case_identity.empty
    ),
    "shared_case_projects_match": (
        shared_case_project_matches.all()
    ),
    "shared_case_uuids_match": (
        shared_case_uuid_matches.all()
    ),
}


print()
print("Cross-modality case-identity checks:")

for check_name, passed in cross_modality_case_checks.items():
    print(f"{check_name}: {passed}")

failed_cross_modality_checks = [
    check_name
    for check_name, passed
    in cross_modality_case_checks.items()
    if not passed
]

if failed_cross_modality_checks:
    raise ValueError(
        "Cross-modality case-identity validation failed: "
        + ", ".join(failed_cross_modality_checks)
    )


# -----------------------------------------------------------------------------
# Summarize harmonized modality indexes
# -----------------------------------------------------------------------------

harmonized_index_summary = pd.DataFrame(
    [
        {
            "modality": "RNA-seq",
            "n_files": (
                rna_pairing_index["rna_file_id"]
                .nunique()
            ),
            "n_aliquots": (
                rna_pairing_index["rna_aliquot_uuid"]
                .nunique()
            ),
            "n_samples": (
                rna_pairing_index["sample_submitter_id"]
                .nunique()
            ),
            "n_cases": (
                rna_pairing_index["case_submitter_id"]
                .nunique()
            ),
            "n_projects": (
                rna_pairing_index["project_id"]
                .nunique()
            ),
        },
        {
            "modality": "DNA methylation",
            "n_files": (
                methylation_pairing_index[
                    "methylation_file_id"
                ]
                .nunique()
            ),
            "n_aliquots": (
                methylation_pairing_index[
                    "methylation_aliquot_uuid"
                ]
                .nunique()
            ),
            "n_samples": (
                methylation_pairing_index[
                    "sample_submitter_id"
                ]
                .nunique()
            ),
            "n_cases": (
                methylation_pairing_index[
                    "case_submitter_id"
                ]
                .nunique()
            ),
            "n_projects": (
                methylation_pairing_index["project_id"]
                .nunique()
            ),
        },
    ]
)

case_identity_overlap_summary = pd.DataFrame(
    [
        {
            "rna_cases": (
                rna_case_identity[
                    "case_submitter_id"
                ]
                .nunique()
            ),
            "methylation_cases": (
                methylation_case_identity[
                    "case_submitter_id"
                ]
                .nunique()
            ),
            "shared_case_barcodes": (
                shared_case_identity[
                    "case_submitter_id"
                ]
                .nunique()
            ),
            "shared_project_matches": int(
                shared_case_project_matches.sum()
            ),
            "shared_case_uuid_matches": int(
                shared_case_uuid_matches.sum()
            ),
        }
    ]
)


print()
print("Harmonized modality indexes:")
display(harmonized_index_summary)

print("Cross-modality case identity:")
display(case_identity_overlap_summary)

print("Standardized RNA-seq pairing index:")
print(f"Shape: {rna_pairing_index.shape}")
display(rna_pairing_index.head())

print("Standardized methylation pairing index:")
print(f"Shape: {methylation_pairing_index.shape}")
display(methylation_pairing_index.head())

Pairing-index integrity checks:
rna_expected_row_count: True
methylation_expected_row_count: True
rna_unique_file_ids: True
methylation_unique_file_ids: True
rna_required_identifiers_complete: True
methylation_required_identifiers_complete: True
rna_case_barcode_matches_aliquot: True
rna_sample_barcode_matches_aliquot: True
methylation_case_barcode_matches_aliquot: True
methylation_sample_barcode_matches_aliquot: True
rna_one_project_per_case_barcode: True
methylation_one_project_per_case_barcode: True
rna_one_case_uuid_per_case_barcode: True
methylation_one_case_uuid_per_case_barcode: True

Cross-modality case-identity checks:
shared_case_identity_is_not_empty: True
shared_case_projects_match: True
shared_case_uuids_match: True

Harmonized modality indexes:


,modality,n_files,n_aliquots,n_samples,n_cases,n_projects
0,RNA-seq,10308,10308,10174,10122,33
1,DNA methylation,10897,10703,10656,10610,33


Cross-modality case identity:


,rna_cases,methylation_cases,shared_case_barcodes,shared_project_matches,shared_case_uuid_matches
0,10122,10610,10054,10054,10054


Standardized RNA-seq pairing index:
Shape: (10308, 8)


,project_id,case_submitter_id,sample_submitter_id,rna_case_uuid,rna_aliquot_uuid,rna_aliquot_submitter_id,rna_file_id,rna_file_name
0,TCGA-ACC,TCGA-OR-A5J1,TCGA-OR-A5J1-01A,b3164f7b-c826-4e08-9ee6-8ff96d29b913,37e88158-0743-45b8-87cf-1d7fe878527f,TCGA-OR-A5J1-01A-11R-A29S-07,fe16b2d3-17b0-4e24-ab31-62d2e951b3a2,6bacf042-830f-47dd-bac3-5696aebf2574.rna_seq.a...
1,TCGA-ACC,TCGA-OR-A5J2,TCGA-OR-A5J2-01A,8e7c2e31-d085-4b75-a970-162526dd07a0,421bbd6b-fb2a-4a6d-a298-057c1b65f041,TCGA-OR-A5J2-01A-11R-A29S-07,f11aa62e-5d67-4e49-8f64-5ed663a3b424,06cc0bc6-1b36-4318-99ee-dfc83cc134c0.rna_seq.a...
2,TCGA-ACC,TCGA-OR-A5J3,TCGA-OR-A5J3-01A,dfd687bc-6e69-42f7-af94-d17fc150d1a1,83b02107-0c7c-450d-84aa-9ed4e1d755c5,TCGA-OR-A5J3-01A-11R-A29S-07,3137ba2b-0d6b-4fcc-8512-894fd6a43ff6,171e4e82-93ed-4be7-8b61-7c1df82bde0f.rna_seq.a...
3,TCGA-ACC,TCGA-OR-A5J5,TCGA-OR-A5J5-01A,802dbd0d-ef07-4c91-ab8d-1dd39532e947,f72bfbe6-411d-412e-aaab-1a2414e544ec,TCGA-OR-A5J5-01A-11R-A29S-07,b8abd8a3-1525-4bf5-b083-eddefa3ca19e,32d1a985-ad2c-4046-873d-02416f5a68f3.rna_seq.a...
4,TCGA-ACC,TCGA-OR-A5J6,TCGA-OR-A5J6-01A,c8898b42-b704-45a0-9829-144b98f416e0,bb6e0e73-ff48-4627-bce6-f672c06b7de4,TCGA-OR-A5J6-01A-31R-A29S-07,224fa76c-cf8d-48d8-8723-15907a04f2d2,a0169877-dd20-4c7d-826c-bb424199a036.rna_seq.a...


Standardized methylation pairing index:
Shape: (10897, 9)


,project_id,case_submitter_id,sample_submitter_id,methylation_case_uuid,methylation_aliquot_uuid,methylation_aliquot_submitter_id,methylation_file_id,methylation_file_name,methylation_platform
0,TCGA-ACC,TCGA-OR-A5J1,TCGA-OR-A5J1-01A,b3164f7b-c826-4e08-9ee6-8ff96d29b913,8690b8a2-d020-4172-a2a8-1f34bf6cb89c,TCGA-OR-A5J1-01A-11D-A29J-05,060eea58-77fe-4661-81f2-484f145661ac,bb52bb9a-0ff9-4ab3-a91a-dd6082a70905.methylati...,Illumina Human Methylation 450
1,TCGA-ACC,TCGA-OR-A5J2,TCGA-OR-A5J2-01A,8e7c2e31-d085-4b75-a970-162526dd07a0,4a320e08-0101-4ddc-85c6-cbb9e59fb838,TCGA-OR-A5J2-01A-11D-A29J-05,0a887275-50ba-4162-9422-14a3a614d18c,571c403a-8b29-4b29-b74b-eddbc96c90d3.methylati...,Illumina Human Methylation 450
2,TCGA-ACC,TCGA-OR-A5J3,TCGA-OR-A5J3-01A,dfd687bc-6e69-42f7-af94-d17fc150d1a1,8be135eb-a9cd-4f7e-bfa1-a7fae79d57b7,TCGA-OR-A5J3-01A-11D-A29J-05,5f15dfb7-a6b1-49ee-b90b-225866d84de6,74a6331a-773e-4d74-86c6-04a078a78403.methylati...,Illumina Human Methylation 450
3,TCGA-ACC,TCGA-OR-A5J4,TCGA-OR-A5J4-01A,5f3e2974-f1df-47a2-8a8a-29bb525eeef6,6abdb597-d4a4-42e9-8e33-091e2b4fa901,TCGA-OR-A5J4-01A-11D-A29J-05,d2dcc333-1e90-4f63-b9c1-4f970be8056b,05d38477-f2dd-4832-894c-8b0fc8fbcb62.methylati...,Illumina Human Methylation 450
4,TCGA-ACC,TCGA-OR-A5J5,TCGA-OR-A5J5-01A,802dbd0d-ef07-4c91-ab8d-1dd39532e947,1269b1c0-3b7a-4c18-b180-9e644e3c37a7,TCGA-OR-A5J5-01A-11D-A29J-05,548e765a-f532-4316-b87c-93dee6bc2293,2a0baa86-d3d0-4be1-ab6a-bf7a9d820bf4.methylati...,Illumina Human Methylation 450


## Exact-sample cross-modality pairing diagnosis

Case-level overlap establishes that RNA-seq and DNA methylation are available for the same participant, but it does not guarantee that both assays originate from the same TCGA sample or vial.

Candidate molecular pairs are therefore constructed using an exact match on:

`project_id + case_submitter_id + sample_submitter_id`

This preserves the TCGA biospecimen hierarchy while allowing RNA and DNA assays to originate from different modality-specific aliquots of the same sample.

The resulting matches are file-level candidates rather than final analytical selections. Multiplicity may arise when:

- one sample has multiple RNA-seq aliquots;
- one sample has multiple methylation aliquots;
- the same methylation sample was assayed on both HM27 and HM450;
- a case contains more than one exactly matched sample.

The diagnosis distinguishes:

- cases with one exact RNA-seq–methylation file pair;
- cases with multiple exact file-pair candidates;
- cases represented in both modalities but only through different samples;
- cases represented in only one modality.

At this stage:

- no candidate file is preferred or discarded;
- HM27 and HM450 remain distinct;
- no one-sample-per-case rule is applied;
- no output artifact is written.

In [6]:
# =============================================================================
# Diagnose exact-sample RNA-seq–methylation pairing
# =============================================================================

pairing_keys = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
]

case_keys = [
    "project_id",
    "case_submitter_id",
]


# -----------------------------------------------------------------------------
# Summarize modality-specific multiplicity at the exact-sample level
# -----------------------------------------------------------------------------

rna_sample_inventory = (
    rna_pairing_index
    .groupby(
        pairing_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_rna_files=(
            "rna_file_id",
            "nunique",
        ),
        n_rna_aliquots=(
            "rna_aliquot_uuid",
            "nunique",
        ),
    )
)


methylation_sample_inventory = (
    methylation_pairing_index
    .groupby(
        pairing_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
        n_methylation_aliquots=(
            "methylation_aliquot_uuid",
            "nunique",
        ),
        n_methylation_platforms=(
            "methylation_platform",
            "nunique",
        ),
    )
)


sample_modality_inventory = (
    rna_sample_inventory
    .merge(
        methylation_sample_inventory,
        on=pairing_keys,
        how="outer",
        validate="one_to_one",
        indicator="sample_modality_membership",
    )
    .sort_values(
        pairing_keys,
        kind="stable",
        ignore_index=True,
    )
)


sample_count_columns = [
    "n_rna_files",
    "n_rna_aliquots",
    "n_methylation_files",
    "n_methylation_aliquots",
    "n_methylation_platforms",
]

sample_modality_inventory[
    sample_count_columns
] = (
    sample_modality_inventory[
        sample_count_columns
    ]
    .fillna(0)
    .astype("int64")
)


# -----------------------------------------------------------------------------
# Identify exact samples represented in both modalities
# -----------------------------------------------------------------------------

exact_sample_inventory = (
    sample_modality_inventory
    .loc[
        sample_modality_inventory[
            "sample_modality_membership"
        ].eq("both")
    ]
    .copy()
)

exact_sample_inventory[
    "n_candidate_file_pairs"
] = (
    exact_sample_inventory["n_rna_files"]
    * exact_sample_inventory["n_methylation_files"]
)


def classify_sample_pairing_cardinality(row):
    n_rna = row["n_rna_files"]
    n_methylation = row["n_methylation_files"]

    if n_rna == 1 and n_methylation == 1:
        return "one_rna_to_one_methylation"

    if n_rna == 1 and n_methylation > 1:
        return "one_rna_to_multiple_methylation"

    if n_rna > 1 and n_methylation == 1:
        return "multiple_rna_to_one_methylation"

    return "multiple_rna_to_multiple_methylation"


exact_sample_inventory[
    "pairing_cardinality"
] = (
    exact_sample_inventory
    .apply(
        classify_sample_pairing_cardinality,
        axis=1,
    )
    .astype("string")
)

exact_sample_inventory[
    "is_one_to_one_exact_sample"
] = (
    exact_sample_inventory["n_rna_files"].eq(1)
    & exact_sample_inventory[
        "n_methylation_files"
    ].eq(1)
)


# -----------------------------------------------------------------------------
# Construct all exact-sample file-pair candidates
# -----------------------------------------------------------------------------

exact_sample_pair_candidates = (
    rna_pairing_index
    .merge(
        methylation_pairing_index,
        on=pairing_keys,
        how="inner",
        validate="many_to_many",
    )
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "rna_aliquot_submitter_id",
            "methylation_platform",
            "methylation_aliquot_submitter_id",
            "rna_file_id",
            "methylation_file_id",
        ],
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Summarize pairing cardinality at the exact-sample level
# -----------------------------------------------------------------------------

exact_sample_cardinality_summary = (
    exact_sample_inventory
    .groupby(
        "pairing_cardinality",
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "size",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
    )
)


candidate_platform_summary = (
    exact_sample_pair_candidates
    .groupby(
        "methylation_platform",
        as_index=False,
        sort=True,
    )
    .agg(
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
        n_exact_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
    )
)


# -----------------------------------------------------------------------------
# Construct case-level modality and exact-pair inventories
# -----------------------------------------------------------------------------

rna_case_inventory = (
    rna_pairing_index
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_rna_files=(
            "rna_file_id",
            "nunique",
        ),
        n_rna_samples=(
            "sample_submitter_id",
            "nunique",
        ),
    )
)


methylation_case_inventory = (
    methylation_pairing_index
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
        n_methylation_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_methylation_platforms=(
            "methylation_platform",
            "nunique",
        ),
    )
)


exact_case_inventory = (
    exact_sample_inventory
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_sample_matches=(
            "sample_submitter_id",
            "nunique",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
        n_one_to_one_exact_samples=(
            "is_one_to_one_exact_sample",
            "sum",
        ),
    )
)


case_pairing_inventory = (
    rna_case_inventory
    .merge(
        methylation_case_inventory,
        on=case_keys,
        how="outer",
        validate="one_to_one",
        indicator="case_modality_membership",
    )
    .merge(
        exact_case_inventory,
        on=case_keys,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)


case_count_columns = [
    "n_rna_files",
    "n_rna_samples",
    "n_methylation_files",
    "n_methylation_samples",
    "n_methylation_platforms",
    "n_exact_sample_matches",
    "n_candidate_file_pairs",
    "n_one_to_one_exact_samples",
]

case_pairing_inventory[
    case_count_columns
] = (
    case_pairing_inventory[
        case_count_columns
    ]
    .fillna(0)
    .astype("int64")
)


# -----------------------------------------------------------------------------
# Classify mutually exclusive case-level pairing outcomes
# -----------------------------------------------------------------------------

has_rna = (
    case_pairing_inventory["n_rna_files"] > 0
)

has_methylation = (
    case_pairing_inventory[
        "n_methylation_files"
    ] > 0
)

has_both_modalities = (
    has_rna
    & has_methylation
)

n_candidate_pairs = (
    case_pairing_inventory[
        "n_candidate_file_pairs"
    ]
)

case_pairing_status = pd.Series(
    pd.NA,
    index=case_pairing_inventory.index,
    dtype="string",
)

case_pairing_status.loc[
    has_rna
    & ~has_methylation
] = "rna_only"

case_pairing_status.loc[
    ~has_rna
    & has_methylation
] = "methylation_only"

case_pairing_status.loc[
    has_both_modalities
    & n_candidate_pairs.eq(0)
] = "both_modalities_no_exact_sample_match"

case_pairing_status.loc[
    has_both_modalities
    & n_candidate_pairs.eq(1)
] = "one_exact_file_pair_candidate"

case_pairing_status.loc[
    has_both_modalities
    & n_candidate_pairs.gt(1)
] = "multiple_exact_file_pair_candidates"

case_pairing_inventory[
    "case_pairing_status"
] = case_pairing_status


case_pairing_outcome_summary = (
    case_pairing_inventory
    .groupby(
        "case_pairing_status",
        as_index=False,
        sort=False,
    )
    .agg(
        n_cases=(
            "case_submitter_id",
            "size",
        ),
        n_rna_files=(
            "n_rna_files",
            "sum",
        ),
        n_methylation_files=(
            "n_methylation_files",
            "sum",
        ),
        n_exact_sample_matches=(
            "n_exact_sample_matches",
            "sum",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
    )
)


case_outcome_order = {
    "one_exact_file_pair_candidate": 0,
    "multiple_exact_file_pair_candidates": 1,
    "both_modalities_no_exact_sample_match": 2,
    "rna_only": 3,
    "methylation_only": 4,
}

case_pairing_outcome_summary[
    "_sort_order"
] = (
    case_pairing_outcome_summary[
        "case_pairing_status"
    ]
    .map(case_outcome_order)
)

case_pairing_outcome_summary = (
    case_pairing_outcome_summary
    .sort_values(
        "_sort_order",
        kind="stable",
        ignore_index=True,
    )
    .drop(columns="_sort_order")
)


# -----------------------------------------------------------------------------
# Recover sample sets for cases lacking an exact sample match
# -----------------------------------------------------------------------------

rna_case_sample_sets = (
    rna_pairing_index
    .groupby(
        case_keys,
        sort=True,
    )["sample_submitter_id"]
    .agg(
        lambda values: tuple(
            sorted(set(values.astype(str)))
        )
    )
    .reset_index(
        name="rna_sample_submitter_ids"
    )
)


methylation_case_sample_sets = (
    methylation_pairing_index
    .groupby(
        case_keys,
        sort=True,
    )["sample_submitter_id"]
    .agg(
        lambda values: tuple(
            sorted(set(values.astype(str)))
        )
    )
    .reset_index(
        name="methylation_sample_submitter_ids"
    )
)


shared_case_sample_sets = (
    rna_case_sample_sets
    .merge(
        methylation_case_sample_sets,
        on=case_keys,
        how="inner",
        validate="one_to_one",
    )
)

shared_case_sample_sets[
    "has_exact_sample_match"
] = [
    bool(
        set(rna_samples)
        .intersection(methylation_samples)
    )
    for rna_samples, methylation_samples
    in zip(
        shared_case_sample_sets[
            "rna_sample_submitter_ids"
        ],
        shared_case_sample_sets[
            "methylation_sample_submitter_ids"
        ],
    )
]


different_sample_cases = (
    shared_case_sample_sets
    .loc[
        ~shared_case_sample_sets[
            "has_exact_sample_match"
        ]
    ]
    .drop(columns="has_exact_sample_match")
    .copy()
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)

different_sample_cases[
    "rna_sample_submitter_ids"
] = (
    different_sample_cases[
        "rna_sample_submitter_ids"
    ]
    .map(" | ".join)
)

different_sample_cases[
    "methylation_sample_submitter_ids"
] = (
    different_sample_cases[
        "methylation_sample_submitter_ids"
    ]
    .map(" | ".join)
)


# -----------------------------------------------------------------------------
# Validate exact-sample pairing construction
# -----------------------------------------------------------------------------

expected_candidate_pair_count = int(
    exact_sample_inventory[
        "n_candidate_file_pairs"
    ].sum()
)

observed_exact_sample_keys = (
    exact_sample_pair_candidates[
        pairing_keys
    ]
    .drop_duplicates()
)

both_modality_case_count = int(
    has_both_modalities.sum()
)

cases_without_exact_sample_count = int(
    (
        has_both_modalities
        & n_candidate_pairs.eq(0)
    ).sum()
)


exact_pairing_checks = {
    "exact_sample_inventory_is_not_empty": (
        not exact_sample_inventory.empty
    ),
    "candidate_pair_table_is_not_empty": (
        not exact_sample_pair_candidates.empty
    ),
    "candidate_pair_rows_are_unique": (
        not exact_sample_pair_candidates[
            [
                "rna_file_id",
                "methylation_file_id",
            ]
        ]
        .duplicated()
        .any()
    ),
    "candidate_count_matches_sample_products": (
        len(exact_sample_pair_candidates)
        == expected_candidate_pair_count
    ),
    "all_exact_sample_keys_have_candidates": (
        len(observed_exact_sample_keys)
        == len(exact_sample_inventory)
    ),
    "candidate_case_uuids_match": (
        exact_sample_pair_candidates[
            "rna_case_uuid"
        ]
        .eq(
            exact_sample_pair_candidates[
                "methylation_case_uuid"
            ]
        )
        .fillna(False)
        .all()
    ),
    "case_pairing_status_complete": (
        case_pairing_inventory[
            "case_pairing_status"
        ]
        .notna()
        .all()
    ),
    "shared_case_count_matches_identity_check": (
        both_modality_case_count
        == len(shared_case_identity)
    ),
    "different_sample_case_count_matches_status": (
        len(different_sample_cases)
        == cases_without_exact_sample_count
    ),
}


print("Exact-sample pairing integrity checks:")

for check_name, passed in exact_pairing_checks.items():
    print(f"{check_name}: {passed}")

failed_exact_pairing_checks = [
    check_name
    for check_name, passed
    in exact_pairing_checks.items()
    if not passed
]

if failed_exact_pairing_checks:
    raise ValueError(
        "Exact-sample pairing diagnosis failed: "
        + ", ".join(failed_exact_pairing_checks)
    )


# -----------------------------------------------------------------------------
# Report pairing coverage and multiplicity
# -----------------------------------------------------------------------------

sample_pairing_summary = pd.DataFrame(
    [
        {
            "rna_samples": len(
                rna_sample_inventory
            ),
            "methylation_samples": len(
                methylation_sample_inventory
            ),
            "shared_exact_samples": len(
                exact_sample_inventory
            ),
            "candidate_file_pairs": len(
                exact_sample_pair_candidates
            ),
            "shared_cases": (
                both_modality_case_count
            ),
            "shared_cases_with_exact_sample": (
                both_modality_case_count
                - cases_without_exact_sample_count
            ),
            "shared_cases_without_exact_sample": (
                cases_without_exact_sample_count
            ),
        }
    ]
)


print()
print("Exact-sample pairing coverage:")
display(sample_pairing_summary)

print("Exact-sample file-pair cardinality:")
display(exact_sample_cardinality_summary)

print("Candidate methylation-platform representation:")
display(candidate_platform_summary)

print("Case-level pairing outcomes:")
display(case_pairing_outcome_summary)

print("Cases represented by different samples across modalities:")
print(f"Count: {len(different_sample_cases):,}")
display(different_sample_cases.head(20))

print("Exact-sample file-pair candidates:")
print(f"Shape: {exact_sample_pair_candidates.shape}")
display(exact_sample_pair_candidates.head())

Exact-sample pairing integrity checks:
exact_sample_inventory_is_not_empty: True
candidate_pair_table_is_not_empty: True
candidate_pair_rows_are_unique: True
candidate_count_matches_sample_products: True
all_exact_sample_keys_have_candidates: True
candidate_case_uuids_match: True
case_pairing_status_complete: True
shared_case_count_matches_identity_check: True
different_sample_case_count_matches_status: True

Exact-sample pairing coverage:


,rna_samples,methylation_samples,shared_exact_samples,candidate_file_pairs,shared_cases,shared_cases_with_exact_sample,shared_cases_without_exact_sample
0,10174,10656,10082,10398,10054,10038,16


Exact-sample file-pair cardinality:


,pairing_cardinality,n_exact_samples,n_cases,n_candidate_file_pairs
0,multiple_rna_to_multiple_methylation,33,33,132
1,multiple_rna_to_one_methylation,74,74,148
2,one_rna_to_multiple_methylation,139,139,282
3,one_rna_to_one_methylation,9836,9823,9836


Candidate methylation-platform representation:


...

,case_pairing_status,n_cases,n_rna_files,n_methylation_files,n_exact_sample_matches,n_candidate_file_pairs
0,one_exact_file_pair_candidate,9779,9780,9780,9779,9779
1,multiple_exact_file_pair_candidates,259,417,479,303,619
2,both_modalities_no_exact_sample_match,16,16,30,0,0
3,rna_only,68,95,0,0,0
4,methylation_only,556,0,608,0,0


Cases represented by different samples across modalities:
Count: 16


,project_id,case_submitter_id,rna_sample_submitter_ids,methylation_sample_submitter_ids
0,TCGA-BRCA,TCGA-AC-A3QQ,TCGA-AC-A3QQ-01B,TCGA-AC-A3QQ-01A
1,TCGA-LAML,TCGA-AB-2811,TCGA-AB-2811-03B,TCGA-AB-2811-03A
2,TCGA-LAML,TCGA-AB-2841,TCGA-AB-2841-03B,TCGA-AB-2841-03A
3,TCGA-LAML,TCGA-AB-2845,TCGA-AB-2845-03B,TCGA-AB-2845-03A
4,TCGA-LAML,TCGA-AB-2888,TCGA-AB-2888-03B,TCGA-AB-2888-03A
5,TCGA-LAML,TCGA-AB-2896,TCGA-AB-2896-03B,TCGA-AB-2896-03A
6,TCGA-LAML,TCGA-AB-2920,TCGA-AB-2920-03B,TCGA-AB-2920-03A
7,TCGA-LAML,TCGA-AB-2949,TCGA-AB-2949-03B,TCGA-AB-2949-03A
8,TCGA-LAML,TCGA-AB-2952,TCGA-AB-2952-03B,TCGA-AB-2952-03A
9,TCGA-LAML,TCGA-AB-2977,TCGA-AB-2977-03B,TCGA-AB-2977-03A


Exact-sample file-pair candidates:
Shape: (10398, 14)


,project_id,case_submitter_id,sample_submitter_id,rna_case_uuid,rna_aliquot_uuid,rna_aliquot_submitter_id,rna_file_id,rna_file_name,methylation_case_uuid,methylation_aliquot_uuid,methylation_aliquot_submitter_id,methylation_file_id,methylation_file_name,methylation_platform
0,TCGA-ACC,TCGA-OR-A5J1,TCGA-OR-A5J1-01A,b3164f7b-c826-4e08-9ee6-8ff96d29b913,37e88158-0743-45b8-87cf-1d7fe878527f,TCGA-OR-A5J1-01A-11R-A29S-07,fe16b2d3-17b0-4e24-ab31-62d2e951b3a2,6bacf042-830f-47dd-bac3-5696aebf2574.rna_seq.a...,b3164f7b-c826-4e08-9ee6-8ff96d29b913,8690b8a2-d020-4172-a2a8-1f34bf6cb89c,TCGA-OR-A5J1-01A-11D-A29J-05,060eea58-77fe-4661-81f2-484f145661ac,bb52bb9a-0ff9-4ab3-a91a-dd6082a70905.methylati...,Illumina Human Methylation 450
1,TCGA-ACC,TCGA-OR-A5J2,TCGA-OR-A5J2-01A,8e7c2e31-d085-4b75-a970-162526dd07a0,421bbd6b-fb2a-4a6d-a298-057c1b65f041,TCGA-OR-A5J2-01A-11R-A29S-07,f11aa62e-5d67-4e49-8f64-5ed663a3b424,06cc0bc6-1b36-4318-99ee-dfc83cc134c0.rna_seq.a...,8e7c2e31-d085-4b75-a970-162526dd07a0,4a320e08-0101-4ddc-85c6-cbb9e59fb838,TCGA-OR-A5J2-01A-11D-A29J-05,0a887275-50ba-4162-9422-14a3a614d18c,571c403a-8b29-4b29-b74b-eddbc96c90d3.methylati...,Illumina Human Methylation 450
2,TCGA-ACC,TCGA-OR-A5J3,TCGA-OR-A5J3-01A,dfd687bc-6e69-42f7-af94-d17fc150d1a1,83b02107-0c7c-450d-84aa-9ed4e1d755c5,TCGA-OR-A5J3-01A-11R-A29S-07,3137ba2b-0d6b-4fcc-8512-894fd6a43ff6,171e4e82-93ed-4be7-8b61-7c1df82bde0f.rna_seq.a...,dfd687bc-6e69-42f7-af94-d17fc150d1a1,8be135eb-a9cd-4f7e-bfa1-a7fae79d57b7,TCGA-OR-A5J3-01A-11D-A29J-05,5f15dfb7-a6b1-49ee-b90b-225866d84de6,74a6331a-773e-4d74-86c6-04a078a78403.methylati...,Illumina Human Methylation 450
3,TCGA-ACC,TCGA-OR-A5J5,TCGA-OR-A5J5-01A,802dbd0d-ef07-4c91-ab8d-1dd39532e947,f72bfbe6-411d-412e-aaab-1a2414e544ec,TCGA-OR-A5J5-01A-11R-A29S-07,b8abd8a3-1525-4bf5-b083-eddefa3ca19e,32d1a985-ad2c-4046-873d-02416f5a68f3.rna_seq.a...,802dbd0d-ef07-4c91-ab8d-1dd39532e947,1269b1c0-3b7a-4c18-b180-9e644e3c37a7,TCGA-OR-A5J5-01A-11D-A29J-05,548e765a-f532-4316-b87c-93dee6bc2293,2a0baa86-d3d0-4be1-ab6a-bf7a9d820bf4.methylati...,Illumina Human Methylation 450
4,TCGA-ACC,TCGA-OR-A5J6,TCGA-OR-A5J6-01A,c8898b42-b704-45a0-9829-144b98f416e0,bb6e0e73-ff48-4627-bce6-f672c06b7de4,TCGA-OR-A5J6-01A-31R-A29S-07,224fa76c-cf8d-48d8-8723-15907a04f2d2,a0169877-dd20-4c7d-826c-bb424199a036.rna_seq.a...,c8898b42-b704-45a0-9829-144b98f416e0,f000b9b0-df63-4808-a283-0fdb96f231dd,TCGA-OR-A5J6-01A-31D-A29J-05,ef544e69-cb98-46b7-9590-565258bd051e,ae9095ef-9dfa-4c4d-9764-06abea2f1700.methylati...,Illumina Human Methylation 450


## Diagnosis of exact-pair multiplicity

Cases with multiple exact RNA-seq–methylation file-pair candidates are characterized before applying any analytical selection rule.

Multiplicity can arise independently from:

- more than one exactly matched TCGA sample-and-vial barcode within the same case;
- multiple RNA-seq files or aliquots for the same sample-and-vial barcode;
- multiple methylation files, aliquots, or array platforms for the same sample-and-vial barcode.

These situations are not treated as equivalent. In particular, coexistence of HM27 and HM450, repeated aliquots within one platform, and multiple biological sample barcodes require distinct handling.

This diagnostic therefore determines:

- the mutually exclusive sources of case-level ambiguity;
- the number of cases with multiple exact sample matches;
- whether RNA-seq multiplicity reflects repeated files or distinct aliquots;
- whether methylation multiplicity reflects platform overlap, aliquot multiplicity, or both;
- the platform combinations represented among exact matches.

At this stage:

- no RNA-seq or methylation file is selected;
- no platform preference is applied;
- different vials are not incorporated as exact matches;
- no output artifact is written.

In [7]:
# =============================================================================
# Characterize sources of exact-sample pairing multiplicity
# =============================================================================

def join_unique_strings(values):
    return " | ".join(
        sorted(
            set(
                values
                .dropna()
                .astype(str)
            )
        )
    )


# -----------------------------------------------------------------------------
# Add methylation-platform composition to each exact sample
# -----------------------------------------------------------------------------

methylation_sample_platform_inventory = (
    methylation_pairing_index
    .groupby(
        pairing_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_methylation_files_check=(
            "methylation_file_id",
            "nunique",
        ),
        n_methylation_aliquots_check=(
            "methylation_aliquot_uuid",
            "nunique",
        ),
        n_methylation_platforms_check=(
            "methylation_platform",
            "nunique",
        ),
        methylation_platform_combination=(
            "methylation_platform",
            join_unique_strings,
        ),
    )
)


exact_sample_diagnostic = (
    exact_sample_inventory
    .merge(
        methylation_sample_platform_inventory,
        on=pairing_keys,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        pairing_keys,
        kind="stable",
        ignore_index=True,
    )
)


exact_sample_diagnostic[
    "has_multiple_rna_files"
] = (
    exact_sample_diagnostic[
        "n_rna_files"
    ].gt(1)
)

exact_sample_diagnostic[
    "has_multiple_methylation_files"
] = (
    exact_sample_diagnostic[
        "n_methylation_files"
    ].gt(1)
)

exact_sample_diagnostic[
    "has_multiple_methylation_platforms"
] = (
    exact_sample_diagnostic[
        "n_methylation_platforms"
    ].gt(1)
)


# -----------------------------------------------------------------------------
# Summarize ambiguity sources at the case level
# -----------------------------------------------------------------------------

exact_case_diagnostic = (
    exact_sample_diagnostic
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
        n_samples_with_multiple_rna_files=(
            "has_multiple_rna_files",
            "sum",
        ),
        n_samples_with_multiple_methylation_files=(
            "has_multiple_methylation_files",
            "sum",
        ),
        n_samples_with_multiple_methylation_platforms=(
            "has_multiple_methylation_platforms",
            "sum",
        ),
    )
)


ambiguous_case_diagnostic = (
    case_pairing_inventory
    .loc[
        case_pairing_inventory[
            "case_pairing_status"
        ].eq(
            "multiple_exact_file_pair_candidates"
        ),
        case_keys,
    ]
    .merge(
        exact_case_diagnostic,
        on=case_keys,
        how="left",
        validate="one_to_one",
    )
)


def classify_case_ambiguity(row):
    sources = []

    if row["n_exact_samples"] > 1:
        sources.append(
            "multiple_exact_samples"
        )

    if (
        row[
            "n_samples_with_multiple_rna_files"
        ]
        > 0
    ):
        sources.append(
            "rna_file_multiplicity"
        )

    if (
        row[
            "n_samples_with_multiple_methylation_files"
        ]
        > 0
    ):
        sources.append(
            "methylation_file_multiplicity"
        )

    return " + ".join(sources)


ambiguous_case_diagnostic[
    "ambiguity_source"
] = (
    ambiguous_case_diagnostic
    .apply(
        classify_case_ambiguity,
        axis=1,
    )
    .astype("string")
)


case_ambiguity_source_summary = (
    ambiguous_case_diagnostic
    .groupby(
        "ambiguity_source",
        as_index=False,
        sort=True,
    )
    .agg(
        n_cases=(
            "case_submitter_id",
            "size",
        ),
        n_exact_samples=(
            "n_exact_samples",
            "sum",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Summarize methylation-platform combinations
# -----------------------------------------------------------------------------

exact_sample_platform_summary = (
    exact_sample_diagnostic
    .groupby(
        "methylation_platform_combination",
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "size",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_methylation_files=(
            "n_methylation_files",
            "sum",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Diagnose RNA-seq file and aliquot multiplicity
# -----------------------------------------------------------------------------

rna_multiplicity_samples = (
    exact_sample_diagnostic
    .loc[
        exact_sample_diagnostic[
            "has_multiple_rna_files"
        ]
    ]
    .copy()
)


def classify_rna_multiplicity(row):
    n_files = row["n_rna_files"]
    n_aliquots = row["n_rna_aliquots"]

    if n_aliquots == 1:
        return "same_aliquot_multiple_files"

    if n_files == n_aliquots:
        return "one_file_per_multiple_aliquot"

    return "multiple_aliquots_with_repeated_files"


rna_multiplicity_samples[
    "rna_multiplicity_structure"
] = (
    rna_multiplicity_samples
    .apply(
        classify_rna_multiplicity,
        axis=1,
    )
    .astype("string")
)


rna_multiplicity_summary = (
    rna_multiplicity_samples
    .groupby(
        "rna_multiplicity_structure",
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "size",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_rna_files=(
            "n_rna_files",
            "sum",
        ),
        n_rna_aliquots=(
            "n_rna_aliquots",
            "sum",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Diagnose methylation file, aliquot, and platform multiplicity
# -----------------------------------------------------------------------------

methylation_multiplicity_samples = (
    exact_sample_diagnostic
    .loc[
        exact_sample_diagnostic[
            "has_multiple_methylation_files"
        ]
    ]
    .copy()
)


def classify_methylation_multiplicity(row):
    n_aliquots = row[
        "n_methylation_aliquots"
    ]
    n_platforms = row[
        "n_methylation_platforms"
    ]

    if n_aliquots == 1 and n_platforms == 1:
        return (
            "same_aliquot_same_platform_multiple_files"
        )

    if n_aliquots == 1 and n_platforms > 1:
        return (
            "same_aliquot_multiple_platforms"
        )

    if n_aliquots > 1 and n_platforms == 1:
        return (
            "multiple_aliquots_single_platform"
        )

    return "multiple_aliquots_multiple_platforms"


methylation_multiplicity_samples[
    "methylation_multiplicity_structure"
] = (
    methylation_multiplicity_samples
    .apply(
        classify_methylation_multiplicity,
        axis=1,
    )
    .astype("string")
)


methylation_multiplicity_summary = (
    methylation_multiplicity_samples
    .groupby(
        [
            "methylation_multiplicity_structure",
            "methylation_platform_combination",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "size",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_methylation_files=(
            "n_methylation_files",
            "sum",
        ),
        n_methylation_aliquots=(
            "n_methylation_aliquots",
            "sum",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Recover exact-sample sets for cases with more than one match
# -----------------------------------------------------------------------------

exact_sample_descriptor_table = (
    exact_sample_diagnostic
    .copy()
)

exact_sample_descriptor_table[
    "exact_sample_descriptor"
] = (
    exact_sample_descriptor_table[
        "sample_submitter_id"
    ].astype("string")
    + " ["
    + exact_sample_descriptor_table[
        "pairing_cardinality"
    ].astype("string")
    + "; "
    + exact_sample_descriptor_table[
        "methylation_platform_combination"
    ].astype("string")
    + "]"
)


case_exact_sample_descriptors = (
    exact_sample_descriptor_table
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        exact_sample_candidates=(
            "exact_sample_descriptor",
            lambda values: " || ".join(
                values.astype(str)
            ),
        )
    )
)


multiple_exact_sample_cases = (
    exact_case_diagnostic
    .loc[
        exact_case_diagnostic[
            "n_exact_samples"
        ].gt(1)
    ]
    .merge(
        ambiguous_case_diagnostic[
            case_keys
            + [
                "ambiguity_source",
            ]
        ],
        on=case_keys,
        how="left",
        validate="one_to_one",
    )
    .merge(
        case_exact_sample_descriptors,
        on=case_keys,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Validate multiplicity diagnosis
# -----------------------------------------------------------------------------

expected_ambiguous_case_count = int(
    case_pairing_inventory[
        "case_pairing_status"
    ]
    .eq(
        "multiple_exact_file_pair_candidates"
    )
    .sum()
)

expected_ambiguous_candidate_count = int(
    case_pairing_inventory
    .loc[
        case_pairing_inventory[
            "case_pairing_status"
        ].eq(
            "multiple_exact_file_pair_candidates"
        ),
        "n_candidate_file_pairs",
    ]
    .sum()
)


multiplicity_checks = {
    "exact_sample_row_count_preserved": (
        len(exact_sample_diagnostic)
        == len(exact_sample_inventory)
    ),
    "methylation_file_counts_match": (
        exact_sample_diagnostic[
            "n_methylation_files"
        ]
        .eq(
            exact_sample_diagnostic[
                "n_methylation_files_check"
            ]
        )
        .all()
    ),
    "methylation_aliquot_counts_match": (
        exact_sample_diagnostic[
            "n_methylation_aliquots"
        ]
        .eq(
            exact_sample_diagnostic[
                "n_methylation_aliquots_check"
            ]
        )
        .all()
    ),
    "methylation_platform_counts_match": (
        exact_sample_diagnostic[
            "n_methylation_platforms"
        ]
        .eq(
            exact_sample_diagnostic[
                "n_methylation_platforms_check"
            ]
        )
        .all()
    ),
    "candidate_pair_total_preserved": (
        exact_sample_diagnostic[
            "n_candidate_file_pairs"
        ].sum()
        == len(exact_sample_pair_candidates)
    ),
    "ambiguous_case_count_matches_status": (
        len(ambiguous_case_diagnostic)
        == expected_ambiguous_case_count
    ),
    "ambiguous_candidate_count_matches_status": (
        ambiguous_case_diagnostic[
            "n_candidate_file_pairs"
        ].sum()
        == expected_ambiguous_candidate_count
    ),
    "all_ambiguous_cases_have_a_source": (
        ambiguous_case_diagnostic[
            "ambiguity_source"
        ]
        .str.len()
        .gt(0)
        .all()
    ),
    "all_ambiguous_cases_have_multiple_pairs": (
        ambiguous_case_diagnostic[
            "n_candidate_file_pairs"
        ]
        .gt(1)
        .all()
    ),
}


print("Multiplicity-diagnosis integrity checks:")

for check_name, passed in multiplicity_checks.items():
    print(f"{check_name}: {passed}")

failed_multiplicity_checks = [
    check_name
    for check_name, passed
    in multiplicity_checks.items()
    if not passed
]

if failed_multiplicity_checks:
    raise ValueError(
        "Pairing-multiplicity diagnosis failed: "
        + ", ".join(failed_multiplicity_checks)
    )


# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

pairing_multiplicity_overview = pd.DataFrame(
    [
        {
            "n_exact_samples": (
                len(exact_sample_diagnostic)
            ),
            "n_exact_cases": (
                len(exact_case_diagnostic)
            ),
            "n_additional_exact_sample_matches": (
                len(exact_sample_diagnostic)
                - len(exact_case_diagnostic)
            ),
            "n_exact_samples_with_file_multiplicity": int(
                (
                    exact_sample_diagnostic[
                        "has_multiple_rna_files"
                    ]
                    | exact_sample_diagnostic[
                        "has_multiple_methylation_files"
                    ]
                ).sum()
            ),
            "n_cases_with_multiple_exact_samples": (
                len(multiple_exact_sample_cases)
            ),
            "n_cases_with_multiple_candidate_pairs": (
                len(ambiguous_case_diagnostic)
            ),
            "n_candidate_pairs_in_ambiguous_cases": (
                expected_ambiguous_candidate_count
            ),
        }
    ]
)


ambiguous_exact_samples = (
    exact_sample_diagnostic
    .loc[
        exact_sample_diagnostic[
            "has_multiple_rna_files"
        ]
        | exact_sample_diagnostic[
            "has_multiple_methylation_files"
        ],
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "n_rna_files",
            "n_rna_aliquots",
            "n_methylation_files",
            "n_methylation_aliquots",
            "n_methylation_platforms",
            "methylation_platform_combination",
            "n_candidate_file_pairs",
            "pairing_cardinality",
        ],
    ]
    .sort_values(
        pairing_keys,
        kind="stable",
        ignore_index=True,
    )
)


print()
print("Pairing multiplicity overview:")
display(pairing_multiplicity_overview)

print("Case-level ambiguity sources:")
display(case_ambiguity_source_summary)

print("Exact-sample methylation-platform combinations:")
display(exact_sample_platform_summary)

print("RNA-seq multiplicity structure:")
display(rna_multiplicity_summary)

print("Methylation multiplicity structure:")
display(methylation_multiplicity_summary)

print("Cases with more than one exact sample match:")
print(f"Count: {len(multiple_exact_sample_cases):,}")
display(multiple_exact_sample_cases.head(30))

print("Exact samples with file-level multiplicity:")
print(f"Count: {len(ambiguous_exact_samples):,}")
display(ambiguous_exact_samples.head(30))

Multiplicity-diagnosis integrity checks:
exact_sample_row_count_preserved: True
methylation_file_counts_match: True
methylation_aliquot_counts_match: True
methylation_platform_counts_match: True
candidate_pair_total_preserved: True
ambiguous_case_count_matches_status: True
ambiguous_candidate_count_matches_status: True
all_ambiguous_cases_have_a_source: True
all_ambiguous_cases_have_multiple_pairs: True

Pairing multiplicity overview:


,n_exact_samples,n_exact_cases,n_additional_exact_sample_matches,n_exact_samples_with_file_multiplicity,n_cases_with_multiple_exact_samples,n_cases_with_multiple_candidate_pairs,n_candidate_pairs_in_ambiguous_cases
0,10082,10038,44,246,44,259,619


Case-level ambiguity sources:


,ambiguity_source,n_cases,n_exact_samples,n_candidate_file_pairs
0,methylation_file_multiplicity,139,139,282
1,multiple_exact_samples,13,26,26
2,multiple_exact_samples + rna_file_multiplicity...,31,62,155
3,rna_file_multiplicity,74,74,148
4,rna_file_multiplicity + methylation_file_multi...,2,2,8


Exact-sample methylation-platform combinations:


...

,methylation_multiplicity_structure,methylation_platform_combination,n_exact_samples,n_cases,n_methylation_files,n_methylation_aliquots,n_candidate_file_pairs
0,multiple_aliquots_multiple_platforms,Illumina Human Methylation 27 | Illumina Human...,16,16,32,32,56
1,multiple_aliquots_single_platform,Illumina Human Methylation 27,2,2,8,8,10
2,multiple_aliquots_single_platform,Illumina Human Methylation 450,20,20,40,40,80
3,same_aliquot_multiple_platforms,Illumina Human Methylation 27 | Illumina Human...,134,134,268,134,268


Cases with more than one exact sample match:
Count: 44


,project_id,case_submitter_id,n_exact_samples,n_candidate_file_pairs,n_samples_with_multiple_rna_files,n_samples_with_multiple_methylation_files,n_samples_with_multiple_methylation_platforms,ambiguity_source,exact_sample_candidates
0,TCGA-BLCA,TCGA-BL-A0C8,2,5,1,1,0,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A0C8-01A [multiple_rna_to_multiple_met...
1,TCGA-BLCA,TCGA-BL-A13I,2,5,1,1,0,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A13I-01A [multiple_rna_to_multiple_met...
2,TCGA-BLCA,TCGA-BL-A13J,2,5,1,1,0,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A13J-01A [multiple_rna_to_multiple_met...
3,TCGA-BRCA,TCGA-A7-A0DC,2,2,0,0,0,multiple_exact_samples,TCGA-A7-A0DC-01A [one_rna_to_one_methylation; ...
4,TCGA-BRCA,TCGA-A7-A13G,2,2,0,0,0,multiple_exact_samples,TCGA-A7-A13G-01A [one_rna_to_one_methylation; ...
5,TCGA-BRCA,TCGA-A7-A26E,2,5,1,1,0,multiple_exact_samples + rna_file_multiplicity...,TCGA-A7-A26E-01A [multiple_rna_to_multiple_met...
6,TCGA-BRCA,TCGA-A7-A26F,2,2,0,0,0,multiple_exact_samples,TCGA-A7-A26F-01A [one_rna_to_one_methylation; ...
7,TCGA-BRCA,TCGA-A7-A26J,2,5,1,1,0,multiple_exact_samples + rna_file_multiplicity...,TCGA-A7-A26J-01A [multiple_rna_to_multiple_met...
8,TCGA-BRCA,TCGA-AC-A2QH,2,2,0,0,0,multiple_exact_samples,TCGA-AC-A2QH-01A [one_rna_to_one_methylation; ...
9,TCGA-BRCA,TCGA-AC-A3OD,2,2,0,0,0,multiple_exact_samples,TCGA-AC-A3OD-01A [one_rna_to_one_methylation; ...


Exact samples with file-level multiplicity:
Count: 246


,project_id,case_submitter_id,sample_submitter_id,n_rna_files,n_rna_aliquots,n_methylation_files,n_methylation_aliquots,n_methylation_platforms,methylation_platform_combination,n_candidate_file_pairs,pairing_cardinality
0,TCGA-BLCA,TCGA-BL-A0C8,TCGA-BL-A0C8-01A,2,2,2,2,1,Illumina Human Methylation 450,4,multiple_rna_to_multiple_methylation
1,TCGA-BLCA,TCGA-BL-A13I,TCGA-BL-A13I-01A,2,2,2,2,1,Illumina Human Methylation 450,4,multiple_rna_to_multiple_methylation
2,TCGA-BLCA,TCGA-BL-A13J,TCGA-BL-A13J-01A,2,2,2,2,1,Illumina Human Methylation 450,4,multiple_rna_to_multiple_methylation
3,TCGA-BRCA,TCGA-A7-A0DB,TCGA-A7-A0DB-01A,2,2,1,1,1,Illumina Human Methylation 27,2,multiple_rna_to_one_methylation
4,TCGA-BRCA,TCGA-A7-A13D,TCGA-A7-A13D-01A,2,2,1,1,1,Illumina Human Methylation 450,2,multiple_rna_to_one_methylation
5,TCGA-BRCA,TCGA-A7-A13E,TCGA-A7-A13E-01A,2,2,1,1,1,Illumina Human Methylation 450,2,multiple_rna_to_one_methylation
6,TCGA-BRCA,TCGA-A7-A26E,TCGA-A7-A26E-01A,2,2,2,2,1,Illumina Human Methylation 450,4,multiple_rna_to_multiple_methylation
7,TCGA-BRCA,TCGA-A7-A26J,TCGA-A7-A26J-01A,2,2,2,2,1,Illumina Human Methylation 450,4,multiple_rna_to_multiple_methylation
8,TCGA-COAD,TCGA-A6-2674,TCGA-A6-2674-01A,2,2,1,1,1,Illumina Human Methylation 27,2,multiple_rna_to_one_methylation
9,TCGA-COAD,TCGA-A6-2677,TCGA-A6-2677-01A,2,2,2,2,2,Illumina Human Methylation 27 | Illumina Human...,4,multiple_rna_to_multiple_methylation


In [8]:
# =============================================================================
# Diagnose TCGA biospecimen-portion concordance
# =============================================================================

TCGA_ALIQUOT_BARCODE_PATTERN = (
    r"^(?P<project>TCGA)-"
    r"(?P<tss>[A-Z0-9]{2})-"
    r"(?P<participant>[A-Z0-9]{4})-"
    r"(?P<sample_type_code>[0-9]{2})"
    r"(?P<vial>[A-Z])-"
    r"(?P<portion_number>[0-9]{2})"
    r"(?P<analyte_code>[A-Z])-"
    r"(?P<plate>[A-Z0-9]{4})-"
    r"(?P<center_code>[0-9]{2})$"
)


def extract_tcga_aliquot_components(
    barcode_series,
    prefix,
):
    components = (
        barcode_series
        .astype("string")
        .str.extract(
            TCGA_ALIQUOT_BARCODE_PATTERN,
            expand=True,
        )
        .add_prefix(f"{prefix}_")
    )

    components[
        f"{prefix}_sample_submitter_id_from_aliquot"
    ] = (
        components[f"{prefix}_project"]
        + "-"
        + components[f"{prefix}_tss"]
        + "-"
        + components[f"{prefix}_participant"]
        + "-"
        + components[f"{prefix}_sample_type_code"]
        + components[f"{prefix}_vial"]
    )

    return components


# -----------------------------------------------------------------------------
# Parse RNA-seq and methylation aliquot barcodes
# -----------------------------------------------------------------------------

rna_aliquot_components = (
    extract_tcga_aliquot_components(
        exact_sample_pair_candidates[
            "rna_aliquot_submitter_id"
        ],
        prefix="rna",
    )
)


methylation_aliquot_components = (
    extract_tcga_aliquot_components(
        exact_sample_pair_candidates[
            "methylation_aliquot_submitter_id"
        ],
        prefix="methylation",
    )
)


portion_pair_candidates = pd.concat(
    [
        exact_sample_pair_candidates.copy(),
        rna_aliquot_components,
        methylation_aliquot_components,
    ],
    axis=1,
)


portion_pair_candidates[
    "same_biospecimen_portion"
] = (
    portion_pair_candidates[
        "rna_portion_number"
    ]
    .eq(
        portion_pair_candidates[
            "methylation_portion_number"
        ]
    )
)


portion_pair_candidates[
    "analyte_pair"
] = (
    portion_pair_candidates[
        "rna_analyte_code"
    ].astype("string")
    + " → "
    + portion_pair_candidates[
        "methylation_analyte_code"
    ].astype("string")
)


# -----------------------------------------------------------------------------
# Summarize candidate-pair portion concordance
# -----------------------------------------------------------------------------

portion_candidate_summary = (
    portion_pair_candidates
    .groupby(
        [
            "same_biospecimen_portion",
            "methylation_platform",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
        n_exact_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_rna_files=(
            "rna_file_id",
            "nunique",
        ),
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
    )
)


analyte_pair_summary = (
    portion_pair_candidates
    .groupby(
        [
            "rna_analyte_code",
            "methylation_analyte_code",
            "analyte_pair",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
        n_exact_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
    )
)


# -----------------------------------------------------------------------------
# Summarize portion concordance at the exact-sample level
# -----------------------------------------------------------------------------

exact_sample_portion_diagnostic = (
    portion_pair_candidates
    .groupby(
        pairing_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
        n_same_portion_candidate_pairs=(
            "same_biospecimen_portion",
            "sum",
        ),
        n_rna_portions=(
            "rna_portion_number",
            "nunique",
        ),
        n_methylation_portions=(
            "methylation_portion_number",
            "nunique",
        ),
    )
)


exact_sample_portion_diagnostic[
    "has_same_portion_candidate"
] = (
    exact_sample_portion_diagnostic[
        "n_same_portion_candidate_pairs"
    ].gt(0)
)


def classify_portion_resolution(
    n_same_portion_candidates,
):
    if n_same_portion_candidates == 0:
        return "no_same_portion_candidate"

    if n_same_portion_candidates == 1:
        return "one_same_portion_candidate"

    return "multiple_same_portion_candidates"


exact_sample_portion_diagnostic[
    "portion_resolution_status"
] = (
    exact_sample_portion_diagnostic[
        "n_same_portion_candidate_pairs"
    ]
    .map(classify_portion_resolution)
    .astype("string")
)


exact_sample_portion_summary = (
    exact_sample_portion_diagnostic
    .groupby(
        "portion_resolution_status",
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "size",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
        n_same_portion_candidate_pairs=(
            "n_same_portion_candidate_pairs",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Summarize portion concordance at the case level
# -----------------------------------------------------------------------------

exact_case_portion_diagnostic = (
    exact_sample_portion_diagnostic
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "size",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
        n_same_portion_candidate_pairs=(
            "n_same_portion_candidate_pairs",
            "sum",
        ),
        n_exact_samples_with_same_portion_candidate=(
            "has_same_portion_candidate",
            "sum",
        ),
    )
)


ambiguous_case_portion_diagnostic = (
    ambiguous_case_diagnostic[
        case_keys
        + [
            "ambiguity_source",
        ]
    ]
    .merge(
        exact_case_portion_diagnostic,
        on=case_keys,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)


ambiguous_case_portion_diagnostic[
    "portion_resolution_status"
] = (
    ambiguous_case_portion_diagnostic[
        "n_same_portion_candidate_pairs"
    ]
    .map(classify_portion_resolution)
    .astype("string")
)


case_portion_resolution_summary = (
    ambiguous_case_portion_diagnostic
    .groupby(
        [
            "ambiguity_source",
            "portion_resolution_status",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        n_cases=(
            "case_submitter_id",
            "size",
        ),
        n_exact_samples=(
            "n_exact_samples",
            "sum",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
        n_same_portion_candidate_pairs=(
            "n_same_portion_candidate_pairs",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Recover candidate-level details for ambiguous cases
# -----------------------------------------------------------------------------

ambiguous_candidate_portion_details = (
    portion_pair_candidates
    .merge(
        ambiguous_case_diagnostic[
            case_keys
            + [
                "ambiguity_source",
            ]
        ],
        on=case_keys,
        how="inner",
        validate="many_to_one",
    )
    [
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "ambiguity_source",
            "rna_aliquot_submitter_id",
            "rna_portion_number",
            "methylation_aliquot_submitter_id",
            "methylation_portion_number",
            "methylation_platform",
            "same_biospecimen_portion",
        ]
    ]
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "rna_aliquot_submitter_id",
            "methylation_platform",
            "methylation_aliquot_submitter_id",
        ],
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Validate barcode parsing and portion diagnosis
# -----------------------------------------------------------------------------

portion_diagnosis_checks = {
    "candidate_pair_row_count_preserved": (
        len(portion_pair_candidates)
        == len(exact_sample_pair_candidates)
    ),
    "all_rna_aliquot_barcodes_parseable": (
        rna_aliquot_components
        .notna()
        .all()
        .all()
    ),
    "all_methylation_aliquot_barcodes_parseable": (
        methylation_aliquot_components
        .notna()
        .all()
        .all()
    ),
    "rna_sample_barcode_matches_candidate_sample": (
        portion_pair_candidates[
            "rna_sample_submitter_id_from_aliquot"
        ]
        .eq(
            portion_pair_candidates[
                "sample_submitter_id"
            ]
        )
        .all()
    ),
    "methylation_sample_barcode_matches_candidate_sample": (
        portion_pair_candidates[
            "methylation_sample_submitter_id_from_aliquot"
        ]
        .eq(
            portion_pair_candidates[
                "sample_submitter_id"
            ]
        )
        .all()
    ),
    "all_portion_comparisons_defined": (
        portion_pair_candidates[
            "same_biospecimen_portion"
        ]
        .notna()
        .all()
    ),
    "exact_sample_count_preserved": (
        len(exact_sample_portion_diagnostic)
        == len(exact_sample_inventory)
    ),
    "candidate_total_preserved_after_sample_grouping": (
        exact_sample_portion_diagnostic[
            "n_candidate_file_pairs"
        ].sum()
        == len(exact_sample_pair_candidates)
    ),
    "ambiguous_case_count_preserved": (
        len(ambiguous_case_portion_diagnostic)
        == len(ambiguous_case_diagnostic)
    ),
    "ambiguous_candidate_total_preserved": (
        ambiguous_case_portion_diagnostic[
            "n_candidate_file_pairs"
        ].sum()
        == expected_ambiguous_candidate_count
    ),
    "portion_resolution_status_complete": (
        ambiguous_case_portion_diagnostic[
            "portion_resolution_status"
        ]
        .notna()
        .all()
    ),
}


print("Biospecimen-portion integrity checks:")

for check_name, passed in (
    portion_diagnosis_checks.items()
):
    print(f"{check_name}: {passed}")


failed_portion_checks = [
    check_name
    for check_name, passed
    in portion_diagnosis_checks.items()
    if not passed
]

if failed_portion_checks:
    raise ValueError(
        "Biospecimen-portion diagnosis failed: "
        + ", ".join(failed_portion_checks)
    )


# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

print()
print("Candidate-pair portion concordance:")
display(portion_candidate_summary)

print("Analyte-code combinations:")
display(analyte_pair_summary)

print("Exact-sample portion-resolution status:")
display(exact_sample_portion_summary)

print("Portion resolution among ambiguous cases:")
display(case_portion_resolution_summary)

print("Ambiguous cases after portion diagnosis:")
print(
    f"Count: "
    f"{len(ambiguous_case_portion_diagnostic):,}"
)
display(
    ambiguous_case_portion_diagnostic.head(30)
)

print("Candidate-level portion details:")
print(
    f"Shape: "
    f"{ambiguous_candidate_portion_details.shape}"
)
display(
    ambiguous_candidate_portion_details.head(40)
)

Biospecimen-portion integrity checks:
candidate_pair_row_count_preserved: True
all_rna_aliquot_barcodes_parseable: True
all_methylation_aliquot_barcodes_parseable: True
rna_sample_barcode_matches_candidate_sample: True
methylation_sample_barcode_matches_candidate_sample: True
all_portion_comparisons_defined: True
exact_sample_count_preserved: True
candidate_total_preserved_after_sample_grouping: True
ambiguous_case_count_preserved: True
ambiguous_candidate_total_preserved: True
portion_resolution_status_complete: True

Candidate-pair portion concordance:


,same_biospecimen_portion,methylation_platform,n_candidate_file_pairs,n_exact_samples,n_cases,n_rna_files,n_methylation_files
0,False,Illumina Human Methylation 27,72,65,65,68,70
1,False,Illumina Human Methylation 450,7,7,7,7,7
2,True,Illumina Human Methylation 27,1826,1766,1766,1826,1767
3,True,Illumina Human Methylation 450,8493,8397,8355,8453,8417


Analyte-code combinations:


,rna_analyte_code,methylation_analyte_code,analyte_pair,n_candidate_file_pairs,n_exact_samples,n_cases
0,R,D,R → D,10129,9948,9904
1,T,D,T → D,269,135,135


Exact-sample portion-resolution status:


...

,ambiguity_source,portion_resolution_status,n_cases,n_exact_samples,n_candidate_file_pairs,n_same_portion_candidate_pairs
0,methylation_file_multiplicity,multiple_same_portion_candidates,135,135,270,270
1,methylation_file_multiplicity,one_same_portion_candidate,4,4,12,4
2,multiple_exact_samples,multiple_same_portion_candidates,13,26,26,26
3,multiple_exact_samples + rna_file_multiplicity...,multiple_same_portion_candidates,31,62,155,153
4,rna_file_multiplicity,multiple_same_portion_candidates,72,72,144,144
5,rna_file_multiplicity,no_same_portion_candidate,1,1,2,0
6,rna_file_multiplicity,one_same_portion_candidate,1,1,2,1
7,rna_file_multiplicity + methylation_file_multi...,multiple_same_portion_candidates,2,2,8,6


Ambiguous cases after portion diagnosis:
Count: 259


,project_id,case_submitter_id,ambiguity_source,n_exact_samples,n_candidate_file_pairs,n_same_portion_candidate_pairs,n_exact_samples_with_same_portion_candidate,portion_resolution_status
0,TCGA-BLCA,TCGA-BL-A0C8,multiple_exact_samples + rna_file_multiplicity...,2,5,5,2,multiple_same_portion_candidates
1,TCGA-BLCA,TCGA-BL-A13I,multiple_exact_samples + rna_file_multiplicity...,2,5,5,2,multiple_same_portion_candidates
2,TCGA-BLCA,TCGA-BL-A13J,multiple_exact_samples + rna_file_multiplicity...,2,5,5,2,multiple_same_portion_candidates
3,TCGA-BRCA,TCGA-A7-A0DB,rna_file_multiplicity,1,2,2,1,multiple_same_portion_candidates
4,TCGA-BRCA,TCGA-A7-A0DC,multiple_exact_samples,2,2,2,2,multiple_same_portion_candidates
5,TCGA-BRCA,TCGA-A7-A13D,rna_file_multiplicity,1,2,2,1,multiple_same_portion_candidates
6,TCGA-BRCA,TCGA-A7-A13E,rna_file_multiplicity,1,2,2,1,multiple_same_portion_candidates
7,TCGA-BRCA,TCGA-A7-A13G,multiple_exact_samples,2,2,2,2,multiple_same_portion_candidates
8,TCGA-BRCA,TCGA-A7-A26E,multiple_exact_samples + rna_file_multiplicity...,2,5,5,2,multiple_same_portion_candidates
9,TCGA-BRCA,TCGA-A7-A26F,multiple_exact_samples,2,2,2,2,multiple_same_portion_candidates


Candidate-level portion details:
Shape: (619, 10)


,project_id,case_submitter_id,sample_submitter_id,ambiguity_source,rna_aliquot_submitter_id,rna_portion_number,methylation_aliquot_submitter_id,methylation_portion_number,methylation_platform,same_biospecimen_portion
0,TCGA-BLCA,TCGA-BL-A0C8,TCGA-BL-A0C8-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A0C8-01A-11R-A10U-07,11,TCGA-BL-A0C8-01A-11D-A10W-05,11,Illumina Human Methylation 450,True
1,TCGA-BLCA,TCGA-BL-A0C8,TCGA-BL-A0C8-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A0C8-01A-11R-A10U-07,11,TCGA-BL-A0C8-01A-11D-A276-05,11,Illumina Human Methylation 450,True
2,TCGA-BLCA,TCGA-BL-A0C8,TCGA-BL-A0C8-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A0C8-01A-11R-A277-07,11,TCGA-BL-A0C8-01A-11D-A10W-05,11,Illumina Human Methylation 450,True
3,TCGA-BLCA,TCGA-BL-A0C8,TCGA-BL-A0C8-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A0C8-01A-11R-A277-07,11,TCGA-BL-A0C8-01A-11D-A276-05,11,Illumina Human Methylation 450,True
4,TCGA-BLCA,TCGA-BL-A0C8,TCGA-BL-A0C8-01B,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A0C8-01B-04R-A277-07,04,TCGA-BL-A0C8-01B-04D-A276-05,04,Illumina Human Methylation 450,True
5,TCGA-BLCA,TCGA-BL-A13I,TCGA-BL-A13I-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A13I-01A-11R-A13Y-07,11,TCGA-BL-A13I-01A-11D-A13Z-05,11,Illumina Human Methylation 450,True
6,TCGA-BLCA,TCGA-BL-A13I,TCGA-BL-A13I-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A13I-01A-11R-A13Y-07,11,TCGA-BL-A13I-01A-11D-A276-05,11,Illumina Human Methylation 450,True
7,TCGA-BLCA,TCGA-BL-A13I,TCGA-BL-A13I-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A13I-01A-11R-A277-07,11,TCGA-BL-A13I-01A-11D-A13Z-05,11,Illumina Human Methylation 450,True
8,TCGA-BLCA,TCGA-BL-A13I,TCGA-BL-A13I-01A,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A13I-01A-11R-A277-07,11,TCGA-BL-A13I-01A-11D-A276-05,11,Illumina Human Methylation 450,True
9,TCGA-BLCA,TCGA-BL-A13I,TCGA-BL-A13I-01B,multiple_exact_samples + rna_file_multiplicity...,TCGA-BL-A13I-01B-04R-A277-07,04,TCGA-BL-A13I-01B-04D-A276-05,04,Illumina Human Methylation 450,True


## Primary cross-modality candidate eligibility

The primary integrated cohort applies the following biospecimen and platform eligibility policy:

1. RNA-seq and methylation files must correspond to the same TCGA sample and vial.
2. Their aliquot barcodes must identify the same biospecimen portion.
3. When both HM27 and HM450 methylation files are available for the same sample, vial, and portion, HM450 is prioritized because of its broader CpG coverage.
4. HM27 remains eligible when no HM450 file exists for the corresponding biospecimen unit.

These rules define candidate eligibility only. They do not resolve:

- multiple RNA-seq files from the same biospecimen;
- multiple methylation files within the retained platform;
- cases with more than one eligible exact sample;
- other file-level multiplicities requiring modality-specific quality control.

Those remaining decisions are deferred to the RNA-seq and methylation QC stages. No final paired cohort is frozen and no output artifact is written at this stage.

In [10]:
# =============================================================================
# Apply primary biospecimen and methylation-platform eligibility policy
# =============================================================================

HM27_PLATFORM = "Illumina Human Methylation 27"
HM450_PLATFORM = "Illumina Human Methylation 450"

BIOSPECIMEN_AND_PLATFORM_ELIGIBLE = (
    "eligible_after_biospecimen_and_platform_policy"
)

DIFFERENT_PORTION_EXCLUSION = (
    "excluded_different_biospecimen_portion"
)

HM27_PLATFORM_DEPRIORITIZATION = (
    "excluded_hm27_when_hm450_available"
)


# -----------------------------------------------------------------------------
# Initialize candidate-policy inventory
# -----------------------------------------------------------------------------

pair_policy_inventory = (
    portion_pair_candidates
    .copy()
    .reset_index(drop=True)
)

biospecimen_unit_keys = (
    pairing_keys
    + [
        "rna_portion_number",
    ]
)


observed_methylation_platforms = set(
    pair_policy_inventory[
        "methylation_platform"
    ]
    .dropna()
    .unique()
)

expected_methylation_platforms = {
    HM27_PLATFORM,
    HM450_PLATFORM,
}

unexpected_methylation_platforms = (
    observed_methylation_platforms
    - expected_methylation_platforms
)

if unexpected_methylation_platforms:
    raise ValueError(
        "Unexpected methylation platforms detected: "
        + ", ".join(
            sorted(unexpected_methylation_platforms)
        )
    )


# -----------------------------------------------------------------------------
# Detect HM450 availability within each same-portion biospecimen unit
# -----------------------------------------------------------------------------

same_portion_mask = (
    pair_policy_inventory[
        "same_biospecimen_portion"
    ]
    .eq(True)
)

pair_policy_inventory[
    "hm450_available_within_biospecimen_unit"
] = False


same_portion_candidates = (
    pair_policy_inventory
    .loc[same_portion_mask]
    .copy()
)


hm450_availability = (
    same_portion_candidates
    .groupby(
        biospecimen_unit_keys,
        sort=False,
        dropna=False,
    )[
        "methylation_platform"
    ]
    .transform(
        lambda platforms: (
            platforms.eq(HM450_PLATFORM).any()
        )
    )
)


pair_policy_inventory.loc[
    same_portion_candidates.index,
    "hm450_available_within_biospecimen_unit",
] = hm450_availability


# -----------------------------------------------------------------------------
# Classify each candidate pair
# -----------------------------------------------------------------------------

pair_policy_inventory[
    "primary_pair_policy_status"
] = pd.Series(
    BIOSPECIMEN_AND_PLATFORM_ELIGIBLE,
    index=pair_policy_inventory.index,
    dtype="string",
)


pair_policy_inventory.loc[
    ~same_portion_mask,
    "primary_pair_policy_status",
] = DIFFERENT_PORTION_EXCLUSION


hm27_deprioritized_mask = (
    same_portion_mask
    & pair_policy_inventory[
        "methylation_platform"
    ].eq(HM27_PLATFORM)
    & pair_policy_inventory[
        "hm450_available_within_biospecimen_unit"
    ]
)


pair_policy_inventory.loc[
    hm27_deprioritized_mask,
    "primary_pair_policy_status",
] = HM27_PLATFORM_DEPRIORITIZATION


eligible_pair_mask = (
    pair_policy_inventory[
        "primary_pair_policy_status"
    ].eq(BIOSPECIMEN_AND_PLATFORM_ELIGIBLE)
)


primary_eligible_pair_candidates = (
    pair_policy_inventory
    .loc[eligible_pair_mask]
    .copy()
    .reset_index(drop=True)
)


policy_excluded_pair_candidates = (
    pair_policy_inventory
    .loc[~eligible_pair_mask]
    .copy()
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# Identify cases removed because no same-portion candidate exists
# -----------------------------------------------------------------------------

pre_policy_cases = (
    pair_policy_inventory[
        case_keys
    ]
    .drop_duplicates()
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)


post_policy_cases = (
    primary_eligible_pair_candidates[
        case_keys
    ]
    .drop_duplicates()
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)


cases_removed_by_portion_policy = (
    pre_policy_cases
    .merge(
        post_policy_cases,
        on=case_keys,
        how="left",
        indicator=True,
        validate="one_to_one",
    )
    .loc[
        lambda frame: frame["_merge"].eq(
            "left_only"
        ),
        case_keys,
    ]
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)


cases_with_same_portion_candidate = (
    pair_policy_inventory
    .loc[
        same_portion_mask,
        case_keys,
    ]
    .drop_duplicates()
)


expected_removed_cases = (
    pre_policy_cases
    .merge(
        cases_with_same_portion_candidate,
        on=case_keys,
        how="left",
        indicator=True,
        validate="one_to_one",
    )
    .loc[
        lambda frame: frame["_merge"].eq(
            "left_only"
        ),
        case_keys,
    ]
    .sort_values(
        case_keys,
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Diagnose remaining candidate multiplicity
# -----------------------------------------------------------------------------

primary_case_candidate_diagnostic = (
    primary_eligible_pair_candidates
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
        n_rna_files=(
            "rna_file_id",
            "nunique",
        ),
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
        n_retained_platforms=(
            "methylation_platform",
            "nunique",
        ),
    )
)


primary_case_candidate_diagnostic[
    "candidate_resolution_status"
] = pd.Series(
    "single_eligible_candidate",
    index=primary_case_candidate_diagnostic.index,
    dtype="string",
)


primary_case_candidate_diagnostic.loc[
    primary_case_candidate_diagnostic[
        "n_candidate_file_pairs"
    ].gt(1),
    "candidate_resolution_status",
] = "multiple_eligible_candidates"


candidate_resolution_summary = (
    primary_case_candidate_diagnostic
    .groupby(
        "candidate_resolution_status",
        as_index=False,
        sort=True,
    )
    .agg(
        n_cases=(
            "case_submitter_id",
            "size",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
        n_rna_files=(
            "n_rna_files",
            "sum",
        ),
        n_methylation_files=(
            "n_methylation_files",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Summarize policy effects
# -----------------------------------------------------------------------------

pair_policy_status_summary = (
    pair_policy_inventory
    .groupby(
        "primary_pair_policy_status",
        as_index=False,
        sort=True,
    )
    .agg(
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_exact_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_rna_files=(
            "rna_file_id",
            "nunique",
        ),
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
    )
)


retained_platform_summary = (
    primary_eligible_pair_candidates
    .groupby(
        "methylation_platform",
        as_index=False,
        sort=True,
    )
    .agg(
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
        n_cases=(
            "case_submitter_id",
            "nunique",
        ),
        n_exact_samples=(
            "sample_submitter_id",
            "nunique",
        ),
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
    )
)


# -----------------------------------------------------------------------------
# Validate policy application
# -----------------------------------------------------------------------------

same_portion_units_before_platform_policy = (
    pair_policy_inventory
    .loc[
        same_portion_mask,
        biospecimen_unit_keys,
    ]
    .drop_duplicates()
)


retained_units_after_platform_policy = (
    primary_eligible_pair_candidates[
        biospecimen_unit_keys
    ]
    .drop_duplicates()
)


unit_preservation_check = (
    same_portion_units_before_platform_policy
    .merge(
        retained_units_after_platform_policy,
        on=biospecimen_unit_keys,
        how="outer",
        indicator=True,
        validate="one_to_one",
    )[
        "_merge"
    ]
    .eq("both")
    .all()
)


policy_checks = {
    "candidate_rows_fully_partitioned": (
        len(pair_policy_inventory)
        == (
            len(primary_eligible_pair_candidates)
            + len(policy_excluded_pair_candidates)
        )
    ),
    "policy_status_complete": (
        pair_policy_inventory[
            "primary_pair_policy_status"
        ]
        .notna()
        .all()
    ),
    "all_retained_pairs_share_the_same_portion": (
        primary_eligible_pair_candidates[
            "same_biospecimen_portion"
        ]
        .eq(True)
        .all()
    ),
    "all_different_portion_pairs_excluded": (
        pair_policy_inventory
        .loc[
            ~same_portion_mask,
            "primary_pair_policy_status",
        ]
        .eq(DIFFERENT_PORTION_EXCLUSION)
        .all()
    ),
    "only_hm27_is_platform_deprioritized": (
        pair_policy_inventory
        .loc[
            pair_policy_inventory[
                "primary_pair_policy_status"
            ].eq(HM27_PLATFORM_DEPRIORITIZATION),
            "methylation_platform",
        ]
        .eq(HM27_PLATFORM)
        .all()
    ),
    "retained_hm27_has_no_hm450_alternative": (
        ~primary_eligible_pair_candidates
        .loc[
            primary_eligible_pair_candidates[
                "methylation_platform"
            ].eq(HM27_PLATFORM),
            "hm450_available_within_biospecimen_unit",
        ]
        .any()
    ),
    "hm450_never_deprioritized_by_platform_policy": (
        ~pair_policy_inventory
        .loc[
            pair_policy_inventory[
                "methylation_platform"
            ].eq(HM450_PLATFORM),
            "primary_pair_policy_status",
        ]
        .eq(HM27_PLATFORM_DEPRIORITIZATION)
        .any()
    ),
    "platform_policy_preserves_every_same_portion_unit": (
        unit_preservation_check
    ),
    "removed_cases_are_exactly_cases_without_same_portion_candidates": (
        cases_removed_by_portion_policy.equals(
            expected_removed_cases
        )
    ),
}


print("Primary-pair policy integrity checks:")

for check_name, passed in policy_checks.items():
    print(f"{check_name}: {passed}")


failed_policy_checks = [
    check_name
    for check_name, passed in policy_checks.items()
    if not passed
]

if failed_policy_checks:
    raise ValueError(
        "Primary-pair eligibility policy failed: "
        + ", ".join(failed_policy_checks)
    )


# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

print()
print("Candidate-pair policy outcomes:")
display(pair_policy_status_summary)

print("Retained methylation platforms:")
display(retained_platform_summary)

print("Case-level candidate resolution after policy:")
display(candidate_resolution_summary)

print(
    "Cases removed because no same-portion candidate exists: "
    f"{len(cases_removed_by_portion_policy):,}"
)

display(
    cases_removed_by_portion_policy.head(20)
)

print(
    "Eligible candidate-pair table shape: "
    f"{primary_eligible_pair_candidates.shape}"
)

print()
print(
    "No final file selection was performed and "
    "no output artifact was written."
)

Primary-pair policy integrity checks:
candidate_rows_fully_partitioned: True
policy_status_complete: True
all_retained_pairs_share_the_same_portion: True
all_different_portion_pairs_excluded: True
only_hm27_is_platform_deprioritized: True
retained_hm27_has_no_hm450_alternative: True
hm450_never_deprioritized_by_platform_policy: True
platform_policy_preserves_every_same_portion_unit: True
removed_cases_are_exactly_cases_without_same_portion_candidates: True

Candidate-pair policy outcomes:


,primary_pair_policy_status,n_candidate_file_pairs,n_cases,n_exact_samples,n_rna_files,n_methylation_files
0,eligible_after_biospecimen_and_platform_policy,10162,9973,10017,10122,10038
1,excluded_different_biospecimen_portion,79,72,72,75,77
2,excluded_hm27_when_hm450_available,157,146,146,157,146


Retained methylation platforms:


,methylation_platform,n_candidate_file_pairs,n_cases,n_exact_samples,n_methylation_files
0,Illumina Human Methylation 27,1669,1620,1620,1621
1,Illumina Human Methylation 450,8493,8355,8397,8417


Case-level candidate resolution after policy:


,candidate_resolution_status,n_cases,n_candidate_file_pairs,n_rna_files,n_methylation_files
0,multiple_eligible_candidates,118,307,267,183
1,single_eligible_candidate,9855,9855,9855,9855


Cases removed because no same-portion candidate exists: 65


,project_id,case_submitter_id
0,TCGA-BRCA,TCGA-A7-A0CG
1,TCGA-BRCA,TCGA-E9-A1NC
2,TCGA-COAD,TCGA-AA-A02K
3,TCGA-COAD,TCGA-DM-A1HB
4,TCGA-KIRC,TCGA-A3-3308
5,TCGA-KIRC,TCGA-A3-3311
6,TCGA-KIRC,TCGA-A3-3313
7,TCGA-KIRC,TCGA-A3-3317
8,TCGA-KIRC,TCGA-A3-3319
9,TCGA-KIRC,TCGA-A3-3320


Eligible candidate-pair table shape: (10162, 38)

No final file selection was performed and no output artifact was written.


In [11]:
# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

print()
print("Candidate-pair policy outcomes:")
print(pair_policy_status_summary)

print("Retained methylation platforms:")
print(retained_platform_summary)

print("Case-level candidate resolution after policy:")
print(candidate_resolution_summary)

print(
    "Cases removed because no same-portion candidate exists: "
    f"{len(cases_removed_by_portion_policy):,}"
)

print(
    cases_removed_by_portion_policy.head(20)
)

print(
    "Eligible candidate-pair table shape: "
    f"{primary_eligible_pair_candidates.shape}"
)

print()
print(
    "No final file selection was performed and "
    "no output artifact was written."
)


Candidate-pair policy outcomes:
                       primary_pair_policy_status  n_candidate_file_pairs  \
0  eligible_after_biospecimen_and_platform_policy                   10162   
1          excluded_different_biospecimen_portion                      79   
2              excluded_hm27_when_hm450_available                     157   

   n_cases  n_exact_samples  n_rna_files  n_methylation_files  
0     9973            10017        10122                10038  
1       72               72           75                   77  
2      146              146          157                  146  
Retained methylation platforms:
             methylation_platform  n_candidate_file_pairs  n_cases  \
0   Illumina Human Methylation 27                    1669     1620   
1  Illumina Human Methylation 450                    8493     8355   

   n_exact_samples  n_methylation_files  
0             1620                 1621  
1             8397                 8417  
Case-level candidate resolution a

In [12]:
# =============================================================================
# Diagnose residual multiplicity after biospecimen and platform policy
# =============================================================================

# -----------------------------------------------------------------------------
# Build one row per eligible biospecimen unit
# -----------------------------------------------------------------------------

eligible_biospecimen_unit_diagnostic = (
    primary_eligible_pair_candidates
    .groupby(
        biospecimen_unit_keys,
        as_index=False,
        sort=True,
        dropna=False,
    )
    .agg(
        methylation_portion_number=(
            "methylation_portion_number",
            "first",
        ),
        n_methylation_portion_values=(
            "methylation_portion_number",
            "nunique",
        ),
        methylation_platform=(
            "methylation_platform",
            "first",
        ),
        n_retained_platforms=(
            "methylation_platform",
            "nunique",
        ),
        n_candidate_file_pairs=(
            "methylation_file_id",
            "size",
        ),
        n_rna_files=(
            "rna_file_id",
            "nunique",
        ),
        n_methylation_files=(
            "methylation_file_id",
            "nunique",
        ),
    )
)


eligible_biospecimen_unit_diagnostic[
    "has_rna_file_multiplicity"
] = (
    eligible_biospecimen_unit_diagnostic[
        "n_rna_files"
    ].gt(1)
)


eligible_biospecimen_unit_diagnostic[
    "has_methylation_file_multiplicity"
] = (
    eligible_biospecimen_unit_diagnostic[
        "n_methylation_files"
    ].gt(1)
)


# -----------------------------------------------------------------------------
# Count eligible portions within each exact sample
# -----------------------------------------------------------------------------

eligible_exact_sample_unit_diagnostic = (
    eligible_biospecimen_unit_diagnostic
    .groupby(
        pairing_keys,
        as_index=False,
        sort=True,
        dropna=False,
    )
    .agg(
        n_eligible_biospecimen_units=(
            "rna_portion_number",
            "size",
        ),
    )
)


eligible_exact_sample_unit_diagnostic[
    "has_multiple_eligible_portions"
] = (
    eligible_exact_sample_unit_diagnostic[
        "n_eligible_biospecimen_units"
    ].gt(1)
)


exact_sample_unit_case_summary = (
    eligible_exact_sample_unit_diagnostic
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_exact_samples_with_multiple_eligible_portions=(
            "has_multiple_eligible_portions",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Summarize eligible biospecimen units per case
# -----------------------------------------------------------------------------

biospecimen_unit_case_summary = (
    eligible_biospecimen_unit_diagnostic
    .groupby(
        case_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_eligible_biospecimen_units=(
            "rna_portion_number",
            "size",
        ),
        n_units_with_rna_file_multiplicity=(
            "has_rna_file_multiplicity",
            "sum",
        ),
        n_units_with_methylation_file_multiplicity=(
            "has_methylation_file_multiplicity",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Add residual-ambiguity information to the case-level inventory
# -----------------------------------------------------------------------------

residual_case_candidate_diagnostic = (
    primary_case_candidate_diagnostic
    .merge(
        biospecimen_unit_case_summary,
        on=case_keys,
        how="left",
        validate="one_to_one",
    )
    .merge(
        exact_sample_unit_case_summary,
        on=case_keys,
        how="left",
        validate="one_to_one",
    )
)


residual_case_candidate_diagnostic[
    "has_multiple_retained_platforms"
] = (
    residual_case_candidate_diagnostic[
        "n_retained_platforms"
    ].gt(1)
)


def classify_residual_ambiguity(row):
    if (
        row["candidate_resolution_status"]
        == "single_eligible_candidate"
    ):
        return "single_eligible_candidate"

    ambiguity_components = []

    if row["n_exact_samples"] > 1:
        ambiguity_components.append(
            "multiple_exact_samples"
        )

    if (
        row[
            "n_exact_samples_with_multiple_eligible_portions"
        ]
        > 0
    ):
        ambiguity_components.append(
            "multiple_portions_within_exact_sample"
        )

    if row["n_units_with_rna_file_multiplicity"] > 0:
        ambiguity_components.append(
            "rna_file_multiplicity_within_biospecimen_unit"
        )

    if (
        row[
            "n_units_with_methylation_file_multiplicity"
        ]
        > 0
    ):
        ambiguity_components.append(
            "methylation_file_multiplicity_within_biospecimen_unit"
        )

    if not ambiguity_components:
        return "unclassified_multiple_eligible_candidates"

    return " + ".join(ambiguity_components)


residual_case_candidate_diagnostic[
    "residual_ambiguity_source"
] = (
    residual_case_candidate_diagnostic
    .apply(
        classify_residual_ambiguity,
        axis=1,
    )
    .astype("string")
)


# -----------------------------------------------------------------------------
# Summarize the remaining ambiguous cases
# -----------------------------------------------------------------------------

remaining_ambiguous_cases = (
    residual_case_candidate_diagnostic
    .loc[
        residual_case_candidate_diagnostic[
            "candidate_resolution_status"
        ].eq("multiple_eligible_candidates")
    ]
    .copy()
    .reset_index(drop=True)
)


residual_ambiguity_summary = (
    remaining_ambiguous_cases
    .groupby(
        "residual_ambiguity_source",
        as_index=False,
        sort=True,
    )
    .agg(
        n_cases=(
            "case_submitter_id",
            "size",
        ),
        n_exact_samples=(
            "n_exact_samples",
            "sum",
        ),
        n_biospecimen_units=(
            "n_eligible_biospecimen_units",
            "sum",
        ),
        n_candidate_file_pairs=(
            "n_candidate_file_pairs",
            "sum",
        ),
        n_rna_files=(
            "n_rna_files",
            "sum",
        ),
        n_methylation_files=(
            "n_methylation_files",
            "sum",
        ),
        n_mixed_platform_cases=(
            "has_multiple_retained_platforms",
            "sum",
        ),
    )
)


# -----------------------------------------------------------------------------
# Identify cases retaining different platforms across distinct units
# -----------------------------------------------------------------------------

mixed_platform_cases = (
    residual_case_candidate_diagnostic
    .loc[
        residual_case_candidate_diagnostic[
            "has_multiple_retained_platforms"
        ],
        case_keys,
    ]
    .copy()
)


mixed_platform_biospecimen_units = (
    eligible_biospecimen_unit_diagnostic
    .merge(
        mixed_platform_cases,
        on=case_keys,
        how="inner",
        validate="many_to_one",
    )
    [
        case_keys
        + [
            "sample_submitter_id",
            "rna_portion_number",
            "methylation_platform",
            "n_rna_files",
            "n_methylation_files",
            "n_candidate_file_pairs",
        ]
    ]
    .sort_values(
        case_keys
        + [
            "sample_submitter_id",
            "rna_portion_number",
        ],
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Validate the residual diagnosis
# -----------------------------------------------------------------------------

residual_diagnosis_checks = {
    "all_units_preserve_portion_concordance": (
        eligible_biospecimen_unit_diagnostic[
            "n_methylation_portion_values"
        ].eq(1).all()
        and
        eligible_biospecimen_unit_diagnostic[
            "rna_portion_number"
        ].astype("string").eq(
            eligible_biospecimen_unit_diagnostic[
                "methylation_portion_number"
            ].astype("string")
        ).all()
    ),
    "one_retained_platform_per_biospecimen_unit": (
        eligible_biospecimen_unit_diagnostic[
            "n_retained_platforms"
        ].eq(1).all()
    ),
    "candidate_cartesian_products_preserved": (
        eligible_biospecimen_unit_diagnostic[
            "n_candidate_file_pairs"
        ].eq(
            eligible_biospecimen_unit_diagnostic[
                "n_rna_files"
            ]
            * eligible_biospecimen_unit_diagnostic[
                "n_methylation_files"
            ]
        ).all()
    ),
    "eligible_candidate_total_preserved": (
        eligible_biospecimen_unit_diagnostic[
            "n_candidate_file_pairs"
        ].sum()
        == len(primary_eligible_pair_candidates)
    ),
    "eligible_case_total_preserved": (
        len(residual_case_candidate_diagnostic)
        == primary_eligible_pair_candidates[
            case_keys
        ].drop_duplicates().shape[0]
    ),
    "remaining_ambiguous_case_count_preserved": (
        len(remaining_ambiguous_cases)
        == primary_case_candidate_diagnostic[
            "candidate_resolution_status"
        ].eq("multiple_eligible_candidates").sum()
    ),
    "all_remaining_ambiguities_classified": (
        ~remaining_ambiguous_cases[
            "residual_ambiguity_source"
        ].eq(
            "unclassified_multiple_eligible_candidates"
        ).any()
    ),
}


print("Residual-candidate diagnosis integrity checks:")

for check_name, passed in (
    residual_diagnosis_checks.items()
):
    print(f"{check_name}: {passed}")


failed_residual_checks = [
    check_name
    for check_name, passed
    in residual_diagnosis_checks.items()
    if not passed
]

if failed_residual_checks:
    raise ValueError(
        "Residual-candidate diagnosis failed: "
        + ", ".join(failed_residual_checks)
    )


# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

print()
print("Residual ambiguity after eligibility policy:")
display(residual_ambiguity_summary)

print(
    "Remaining cases with multiple eligible candidates: "
    f"{len(remaining_ambiguous_cases):,}"
)
display(
    remaining_ambiguous_cases.head(30)
)

print("Cases retaining platforms across distinct biospecimen units:")
print(f"Count: {len(mixed_platform_cases):,}")
display(mixed_platform_biospecimen_units)

print()
print(
    "No final candidate selection was performed and "
    "no output artifact was written."
)

Residual-candidate diagnosis integrity checks:
all_units_preserve_portion_concordance: True
one_retained_platform_per_biospecimen_unit: True
candidate_cartesian_products_preserved: True
eligible_candidate_total_preserved: True
eligible_case_total_preserved: True
remaining_ambiguous_case_count_preserved: True
all_remaining_ambiguities_classified: True

Residual ambiguity after eligibility policy:


,residual_ambiguity_source,n_cases,n_exact_samples,n_biospecimen_units,n_candidate_file_pairs,n_rna_files,n_methylation_files,n_mixed_platform_cases
0,multiple_exact_samples,13,26,26,26,26,26,2
1,multiple_exact_samples + rna_file_multiplicity...,11,22,22,33,33,22,0
2,multiple_exact_samples + rna_file_multiplicity...,20,40,40,100,60,60,0
3,multiple_portions_within_exact_sample,1,1,2,2,2,2,0
4,rna_file_multiplicity_within_biospecimen_unit,73,73,73,146,146,73,0


Remaining cases with multiple eligible candidates: 118


,project_id,case_submitter_id,n_exact_samples,n_candidate_file_pairs,n_rna_files,n_methylation_files,n_retained_platforms,candidate_resolution_status,n_eligible_biospecimen_units,n_units_with_rna_file_multiplicity,n_units_with_methylation_file_multiplicity,n_exact_samples_with_multiple_eligible_portions,has_multiple_retained_platforms,residual_ambiguity_source
0,TCGA-BLCA,TCGA-BL-A0C8,2,5,3,3,1,multiple_eligible_candidates,2,1,1,0,False,multiple_exact_samples + rna_file_multiplicity...
1,TCGA-BLCA,TCGA-BL-A13I,2,5,3,3,1,multiple_eligible_candidates,2,1,1,0,False,multiple_exact_samples + rna_file_multiplicity...
2,TCGA-BLCA,TCGA-BL-A13J,2,5,3,3,1,multiple_eligible_candidates,2,1,1,0,False,multiple_exact_samples + rna_file_multiplicity...
3,TCGA-BRCA,TCGA-A7-A0DB,1,2,2,1,1,multiple_eligible_candidates,1,1,0,0,False,rna_file_multiplicity_within_biospecimen_unit
4,TCGA-BRCA,TCGA-A7-A0DC,2,2,2,2,2,multiple_eligible_candidates,2,0,0,0,True,multiple_exact_samples
5,TCGA-BRCA,TCGA-A7-A13D,1,2,2,1,1,multiple_eligible_candidates,1,1,0,0,False,rna_file_multiplicity_within_biospecimen_unit
6,TCGA-BRCA,TCGA-A7-A13E,1,2,2,1,1,multiple_eligible_candidates,1,1,0,0,False,rna_file_multiplicity_within_biospecimen_unit
7,TCGA-BRCA,TCGA-A7-A13G,2,2,2,2,1,multiple_eligible_candidates,2,0,0,0,False,multiple_exact_samples
8,TCGA-BRCA,TCGA-A7-A26E,2,5,3,3,1,multiple_eligible_candidates,2,1,1,0,False,multiple_exact_samples + rna_file_multiplicity...
9,TCGA-BRCA,TCGA-A7-A26F,2,2,2,2,1,multiple_eligible_candidates,2,0,0,0,False,multiple_exact_samples


Cases retaining platforms across distinct biospecimen units:
Count: 2


,project_id,case_submitter_id,sample_submitter_id,rna_portion_number,methylation_platform,n_rna_files,n_methylation_files,n_candidate_file_pairs
0,TCGA-BRCA,TCGA-A7-A0DC,TCGA-A7-A0DC-01A,11,Illumina Human Methylation 27,1,1,1
1,TCGA-BRCA,TCGA-A7-A0DC,TCGA-A7-A0DC-01B,04,Illumina Human Methylation 450,1,1,1
2,TCGA-COAD,TCGA-A6-2672,TCGA-A6-2672-01A,01,Illumina Human Methylation 27,1,1,1
3,TCGA-COAD,TCGA-A6-2672,TCGA-A6-2672-01B,03,Illumina Human Methylation 450,1,1,1



No final candidate selection was performed and no output artifact was written.


In [13]:
# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

print()
print("Residual ambiguity after eligibility policy:")
print(residual_ambiguity_summary)

print(
    "Remaining cases with multiple eligible candidates: "
    f"{len(remaining_ambiguous_cases):,}"
)
print(remaining_ambiguous_cases.head(30))

print("Cases retaining platforms across distinct biospecimen units:")
print(f"Count: {len(mixed_platform_cases):,}")
print(mixed_platform_biospecimen_units)

print()
print(
    "No final candidate selection was performed and "
    "no output artifact was written."
)


Residual ambiguity after eligibility policy:
                           residual_ambiguity_source  n_cases  \
0                             multiple_exact_samples       13   
1  multiple_exact_samples + rna_file_multiplicity...       11   
2  multiple_exact_samples + rna_file_multiplicity...       20   
3              multiple_portions_within_exact_sample        1   
4      rna_file_multiplicity_within_biospecimen_unit       73   

   n_exact_samples  n_biospecimen_units  n_candidate_file_pairs  n_rna_files  \
0               26                   26                      26           26   
1               22                   22                      33           33   
2               40                   40                     100           60   
3                1                    2                       2            2   
4               73                   73                     146          146   

   n_methylation_files  n_mixed_platform_cases  
0                   26           

## Residual candidate ambiguity

Application of the primary biospecimen and platform eligibility policy yielded a single eligible RNA-seq–methylation file pair for 9,855 cases. The remaining 118 cases retained 307 candidate file pairs and were fully classified according to their unresolved multiplicity.

Residual ambiguity comprised:

- 73 cases with multiple RNA-seq files within a single eligible biospecimen unit;
- 13 cases with multiple eligible exact samples and no within-unit file multiplicity;
- 11 cases with multiple eligible exact samples and RNA-seq file multiplicity;
- 20 cases with multiple eligible exact samples and both RNA-seq and methylation file multiplicity;
- 1 case with multiple eligible portions within the same exact sample.

Overall, 104 cases require resolution of RNA-seq file multiplicity, 20 require resolution of methylation file multiplicity, 44 retain multiple exact samples, and 1 retains multiple portions within an exact sample. These categories overlap and therefore are not additive.

Two cases retain HM27 and HM450 candidates across distinct biospecimen units. The HM450-priority rule was intentionally not applied across different samples or portions because platform coverage alone does not justify selecting a different biological unit.

RNA-seq and methylation file multiplicities will be evaluated in their respective modality-specific quality-control stages. Selection among distinct samples or portions will be performed only after both QC outputs are available, using joint cross-modality evidence rather than barcode order, vial designation, UUID, or file ordering.

No final paired cohort was selected at this stage.

In [15]:
# =============================================================================
# Save eligible RNA-seq–methylation candidate-pair inventory
# =============================================================================

ELIGIBLE_PAIR_INVENTORY_DIR = (
    Paths.interim / "metadata"
)

ELIGIBLE_PAIR_INVENTORY_PATH = (
    ELIGIBLE_PAIR_INVENTORY_DIR
    / "tcga_primary_tumor_rnaseq_methylation_eligible_pair_inventory.csv"
)


# -----------------------------------------------------------------------------
# Prepare deterministic output
# -----------------------------------------------------------------------------

required_inventory_columns = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "rna_file_id",
    "methylation_file_id",
    "rna_portion_number",
    "methylation_portion_number",
    "methylation_platform",
    "same_biospecimen_portion",
    "hm450_available_within_biospecimen_unit",
    "primary_pair_policy_status",
]

missing_inventory_columns = [
    column
    for column in required_inventory_columns
    if column not in primary_eligible_pair_candidates.columns
]

if missing_inventory_columns:
    raise ValueError(
        "Required eligible-inventory columns are missing: "
        + ", ".join(missing_inventory_columns)
    )


eligible_pair_inventory = (
    primary_eligible_pair_candidates
    .copy()
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "rna_portion_number",
            "methylation_platform",
            "rna_file_id",
            "methylation_file_id",
        ],
        kind="stable",
        ignore_index=True,
    )
)


# -----------------------------------------------------------------------------
# Validate the artifact before writing
# -----------------------------------------------------------------------------

eligible_inventory_checks = {
    "inventory_is_not_empty": (
        not eligible_pair_inventory.empty
    ),
    "required_columns_are_complete": (
        eligible_pair_inventory[
            required_inventory_columns
        ]
        .notna()
        .all()
        .all()
    ),
    "candidate_file_pairs_are_unique": (
        ~eligible_pair_inventory.duplicated(
            subset=[
                "rna_file_id",
                "methylation_file_id",
            ]
        ).any()
    ),
    "all_pairs_share_the_same_portion": (
        eligible_pair_inventory[
            "same_biospecimen_portion"
        ]
        .eq(True)
        .all()
    ),
    "all_rows_are_policy_eligible": (
        eligible_pair_inventory[
            "primary_pair_policy_status"
        ]
        .eq(
            BIOSPECIMEN_AND_PLATFORM_ELIGIBLE
        )
        .all()
    ),
    "row_count_matches_eligible_candidates": (
        len(eligible_pair_inventory)
        == len(primary_eligible_pair_candidates)
    ),
    "case_count_matches_post_policy_cohort": (
        eligible_pair_inventory[
            case_keys
        ]
        .drop_duplicates()
        .shape[0]
        == len(post_policy_cases)
    ),
}


print("Eligible-inventory pre-write checks:")

for check_name, passed in eligible_inventory_checks.items():
    print(f"{check_name}: {passed}")


failed_inventory_checks = [
    check_name
    for check_name, passed in eligible_inventory_checks.items()
    if not passed
]

if failed_inventory_checks:
    raise ValueError(
        "Eligible candidate-pair inventory validation failed: "
        + ", ".join(failed_inventory_checks)
    )


# -----------------------------------------------------------------------------
# Write the downstream-consumable inventory
# -----------------------------------------------------------------------------

ELIGIBLE_PAIR_INVENTORY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

eligible_pair_inventory.to_csv(
    ELIGIBLE_PAIR_INVENTORY_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Reload and verify the written artifact
# -----------------------------------------------------------------------------

string_column_dtypes = {
    column: "string"
    for column in eligible_pair_inventory.columns
    if (
        pd.api.types.is_object_dtype(
            eligible_pair_inventory[column].dtype
        )
        or pd.api.types.is_string_dtype(
            eligible_pair_inventory[column].dtype
        )
    )
}

reloaded_eligible_pair_inventory = pd.read_csv(
    ELIGIBLE_PAIR_INVENTORY_PATH,
    dtype=string_column_dtypes,
)

written_artifact_checks = {
    "output_file_exists": (
        ELIGIBLE_PAIR_INVENTORY_PATH.is_file()
    ),
    "written_shape_matches_memory": (
        reloaded_eligible_pair_inventory.shape
        == eligible_pair_inventory.shape
    ),
    "written_columns_match_memory": (
        reloaded_eligible_pair_inventory.columns.tolist()
        == eligible_pair_inventory.columns.tolist()
    ),
    "written_file_pair_order_matches_memory": (
        reloaded_eligible_pair_inventory[
            [
                "rna_file_id",
                "methylation_file_id",
            ]
        ]
        .astype("string")
        .equals(
            eligible_pair_inventory[
                [
                    "rna_file_id",
                    "methylation_file_id",
                ]
            ]
            .astype("string")
            .reset_index(drop=True)
        )
    ),
}


print()
print("Written-artifact checks:")

for check_name, passed in written_artifact_checks.items():
    print(f"{check_name}: {passed}")


failed_written_checks = [
    check_name
    for check_name, passed in written_artifact_checks.items()
    if not passed
]

if failed_written_checks:
    raise ValueError(
        "Written eligible inventory failed verification: "
        + ", ".join(failed_written_checks)
    )


# -----------------------------------------------------------------------------
# Report saved artifact
# -----------------------------------------------------------------------------

print()
print("Eligible candidate-pair inventory saved.")
print(f"Path: {project_relative_path(ELIGIBLE_PAIR_INVENTORY_PATH)}")
print(f"Shape: {eligible_pair_inventory.shape}")
print(
    "Cases: "
    f"{eligible_pair_inventory[case_keys].drop_duplicates().shape[0]:,}"
)
print(
    "RNA-seq files: "
    f"{eligible_pair_inventory['rna_file_id'].nunique():,}"
)
print(
    "Methylation files: "
    f"{eligible_pair_inventory['methylation_file_id'].nunique():,}"
)
print(
    "File size: "
    f"{ELIGIBLE_PAIR_INVENTORY_PATH.stat().st_size / (1024 ** 2):.3f} MiB"
)

print()
print(
    "This artifact preserves eligible candidates only; "
    "no final paired-cohort selection was performed."
)

Eligible-inventory pre-write checks:
inventory_is_not_empty: True
required_columns_are_complete: True
candidate_file_pairs_are_unique: True
all_pairs_share_the_same_portion: True
all_rows_are_policy_eligible: True
row_count_matches_eligible_candidates: True
case_count_matches_post_policy_cohort: True

Written-artifact checks:
output_file_exists: True
written_shape_matches_memory: True
written_columns_match_memory: True
written_file_pair_order_matches_memory: True

Eligible candidate-pair inventory saved.
Path: data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_eligible_pair_inventory.csv
Shape: (10162, 38)
Cases: 9,973
RNA-seq files: 10,122
Methylation files: 10,038
File size: 6.463 MiB

This artifact preserves eligible candidates only; no final paired-cohort selection was performed.


## Summary

This notebook constructed a deterministic eligibility inventory of candidate RNA-seq–methylation pairs for TCGA Primary Tumor cases. Pairing was based on the frozen modality-specific file indexes and preserved file-, sample-, portion-, platform-, case-, and project-level provenance.

A total of 10,398 initial candidate file pairs were evaluated. The eligibility policy:

- excluded 79 pairs with discordant RNA-seq and methylation biospecimen portions;
- removed 65 cases for which no same-portion candidate remained;
- deprioritized 157 HM27 candidates when an HM450 file was available for the same biospecimen unit;
- retained HM27 when no same-unit HM450 alternative existed; and
- did not prioritize one platform over another across distinct samples or portions.

After policy application, 10,162 eligible candidate pairs remained, representing:

- 9,973 cases;
- 10,017 exact samples;
- 10,122 RNA-seq files; and
- 10,038 methylation files.

Among the retained cases, 9,855 already had a single eligible file pair. The remaining 118 cases retained 307 candidate pairs and were fully classified according to their unresolved multiplicity:

- 73 cases had RNA-seq file multiplicity within one biospecimen unit;
- 13 retained multiple exact samples without within-unit file multiplicity;
- 11 retained multiple exact samples together with RNA-seq file multiplicity;
- 20 retained multiple exact samples together with RNA-seq and methylation file multiplicity; and
- 1 retained multiple eligible portions within the same exact sample.

Overall, 104 cases require RNA-seq file-level resolution, 20 require methylation file-level resolution, 44 retain multiple exact samples, and 1 retains multiple portions within an exact sample. These latter categories overlap and are therefore not additive.

Two cases retained HM27 and HM450 candidates across different biospecimen units. Both were intentionally preserved because platform coverage alone does not justify selecting a different biological unit.

The validated eligible inventory was written to:

`data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_eligible_pair_inventory.csv`

The artifact contains 10,162 rows and 38 columns. It preserves all candidates that satisfy the current biospecimen and platform policies and serves as the shared input for the modality-specific RNA-seq and methylation quality-control stages.

No final RNA-seq–methylation pair or definitive paired cohort was selected in this notebook. File multiplicity will be resolved through modality-specific QC, followed by joint cross-modality selection among the remaining samples and portions.